# Chapter 8. Numerical Optimal Control — 수치 최적 제어

Rawlings, Mayne, Diehl (2017), **2판 1쇄 제8장, 인쇄 pp.491–600**의 한글 해설·실습입니다.
본문 pp.491–582, 연습문제 pp.583–595, 장 참고문헌 pp.596–600입니다.
원본의 **주요 절 11개와 모든 하위 절, 총 51개 제목**을 같은 순서로 배치합니다.

**학습 목표:** 수치 적분·비선형 방정식·도함수 계산·직접 최적 제어·SQP/내부점법·구조적 선형 대수·실시간 반복을 하나의 계산 흐름으로 연결합니다.

**실행:** Colab에서 `런타임 → 모두 실행`. 공통 준비부터 위에서 아래로 실행합니다. 이전 챕터나 저장소 복제 없이 CPU에서 실행합니다. 최초 패키지·한글 글꼴 설치에는 인터넷이 필요합니다. CasADi는 함수와 도함수를 만들고 SciPy SLSQP가 비선형 최적화를 수행합니다. QP 하위 문제는 CVXPY/Clarabel로 풉니다.

**범위:** 원본 예제 8.1·8.3·8.4·8.5 및 8.9·8.11·8.12의 계산 그래프를 재현합니다. 검증 격자·표본은 실습에 맞게 선택했으며 각 셀에서 범위를 명시합니다. 나머지는 직접 설계한 보충 실습입니다. 교재의 모든 예제·증명·연습문제 해답을 재현하지 않습니다. 수치 검증과 일반적인 안정성·전역 최적성 보장을 구분합니다. 실제 장치의 마감시간이나 통신 지연을 검증하지 않습니다.

**실습별 모델을 구분하세요.**
- §8.2–8.4: 정확도·미분 구조를 확인하는 작은 독립 예제들.
- §8.5: 연속 스칼라 모델, RK4 단일·다중 사격법과 중점 배치법 비교.
- §8.8.2–8.8.5 및 §8.8.7: 2상태 선형 LQ 모델, 리카티·축약·KKT 비교와 추가 입력 제한.
- §8.8.6 및 §8.9: 별도 이산 비선형 모델, iLQR·이동 초기화·Gauss–Newton 실시간 반복.

**기호:** x는 상태, u는 현재 입력, X/U는 예측 열, w는 일반 최적화 변수, λ는 제약 승수입니다. 시간 단계와 최적화 반복 횟수를 구분합니다. LQ·최소제곱 비용에는 표시된 1/2 계수를 적용합니다. 피드백 u=Kx에서 K가 음의 부호를 포함합니다.

[PPT](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter08_numerical_optimal_control/slides/chapter08_slides.pptx) · [PDF](https://github.com/JunsPark00/rawlings-model-predictive-control-notes/blob/main/chapter08_numerical_optimal_control/slides/chapter08_slides.pdf)

## 실습 환경 준비

필요한 패키지와 한글 글꼴을 준비합니다. 각 실습의 잔차·해 오차를 `CHECKS`에 기록합니다.

In [ ]:
import importlib.util, subprocess, sys

packages = {
    "numpy": "numpy>=2,<3",
    "scipy": "scipy>=1.14,<2",
    "matplotlib": "matplotlib>=3.9,<4",
    "cvxpy": "cvxpy>=1.6,<2",
    "osqp": "osqp>=1,<2",
    "casadi": "casadi>=3.7,<4",
}
missing = [
    spec for name, spec in packages.items() if importlib.util.find_spec(name) is None
]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
import numpy as np
import scipy
from scipy.linalg import solve_discrete_are, expm
from scipy.signal import cont2discrete
import matplotlib
import matplotlib.pyplot as plt
import cvxpy as cp
from pathlib import Path
import tempfile
import urllib.request
from matplotlib import font_manager

# Colab과 로컬에서 동일하게 한글 그래프를 표시하기 위한 공개 글꼴.
font_dir = Path(tempfile.gettempdir()) / "mpc_korean_fonts"
font_dir.mkdir(exist_ok=True)
font_path = font_dir / "NanumGothic-Regular.ttf"
if not font_path.exists():
    url = "https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf"
    with urllib.request.urlopen(url, timeout=60) as response:
        font_path.write_bytes(response.read())
font_manager.fontManager.addfont(str(font_path))
korean_font = font_manager.FontProperties(fname=font_path).get_name()

np.set_printoptions(precision=5, suppress=True)
plt.rcParams.update(
    {
        "figure.figsize": (9, 3.8),
        "axes.grid": True,
        "grid.alpha": 0.22,
        "font.size": 11,
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": ["DejaVu Sans", korean_font],
        "mathtext.fontset": "dejavusans",
        "axes.unicode_minus": False,
    }
)
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)
CHECKS = {}


def finish_figure(name):
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{name}.png", dpi=170, bbox_inches="tight")
    plt.show()
    plt.close()

In [ ]:
from scipy.optimize import minimize, root, root_scalar
from scipy.integrate import solve_ivp
from scipy.linalg import block_diag, solve_discrete_are


def record(name, value, tol=None):
    value = float(value)
    if not np.isfinite(value):
        raise ValueError(f"비유한 검증값: {name}")
    CHECKS[name] = value
    if tol is not None:
        assert value < tol, (name, value, tol)


def savefig(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=170, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def positive_integer(value, name):
    if (
        isinstance(value, (bool, np.bool_))
        or not isinstance(value, (int, np.integer))
        or value < 1
    ):
        raise ValueError(f"{name}: 양의 정수가 필요합니다.")
    return int(value)


def finite_scalar(value, name):
    arr = np.asarray(value, dtype=float)
    if arr.shape != () or not np.isfinite(arr):
        raise ValueError(f"{name}: 유한한 스칼라가 필요합니다.")
    return float(arr)


def finite_vector(value, size, name):
    arr = np.asarray(value, dtype=float)
    if arr.shape != (size,) or not np.all(np.isfinite(arr)):
        raise ValueError(f"{name}: 길이 {size}의 유한 벡터가 필요합니다.")
    return arr

## 8.1 Introduction — 개요

**교재 인쇄 pp.491–492**

MPC의 최적화 문제를 수치 계산으로 연결합니다. 모델 적분과 도함수, 최적화 오차를 구분합니다. 시간 제한 안에서 쓸 수 있는 해를 계산합니다.

이 장은 제어기의 안정성 조건을 다시 증명하기보다 앞 장에서 정의한 최적 제어 문제를 실제로 계산하는 방법을 다룹니다. 모델의 연속·이산 표현, 결정 변수 선택, 도함수 계산 및 선형 대수 구조가 최적화 방법의 성능과 정확도에 영향을 줍니다.

### 8.1.1 Discrete Time Optimal Control Problem — 이산 시간 최적 제어 문제

**교재 인쇄 pp.492–493**

상태·입력 열을 유한한 결정 변수로 둡니다. 초기 상태와 동역학을 등식 제약으로 둡니다. 단계 비용과 종단 비용을 합해 최소화합니다.

현재 측정값은 파라미터이고 미래 상태와 입력은 최적화 변수입니다. 경로 제약은 각 시점의 상태·입력에, 종단 제약은 마지막 상태에 적용합니다. 미래 상태를 제거할지 유지할지가 순차 접근과 동시 접근을 나눕니다.

$$
\min_{X,U}\;\sum_{k=0}^{N-1}\ell_k(x_k,u_k)+V_f(x_N)\quad\text{s.t.}\quad x_{k+1}=f_k(x_k,u_k),\;x_0=\bar x
$$

### 8.1.2 Convex Versus Nonconvex Optimization — 볼록·비볼록 최적화

**교재 인쇄 pp.493–496**

볼록 문제는 국소 최소와 전역 최소가 일치합니다. 비선형 동역학 등식은 비볼록성을 만들 수 있습니다. 정상점 잔차만으로 전역 최적성을 주장하지 않습니다.

볼록 비용만으로 전체 문제가 볼록한 것은 아닙니다. 특히 비선형 등식이 만드는 허용 집합을 함께 확인해야 합니다. 보충 예제는 두 시작점에서 서로 다른 정상점으로 수렴합니다. 각 솔버의 도함수 잔차를 확인하고 더 낮은 비용의 해와 구분합니다.

In [ ]:
# 보충 비볼록 함수: 다른 시작점에서 서로 다른 국소 최소에 도달한다.
f = lambda z: (z[0] ** 2 - 1) ** 2 + 0.2 * z[0]
g = lambda z: np.array([4 * z[0] * (z[0] ** 2 - 1) + 0.2])
starts = [-1.5, 1.5]
mins = [minimize(f, [z], jac=g, method="BFGS", options={"gtol": 1e-10}) for z in starts]
assert all(np.linalg.norm(g(s.x)) < 1e-7 for s in mins)
assert abs(mins[0].x[0] - mins[1].x[0]) > 1
record("비볼록 예제 정상점 잔차", max(np.linalg.norm(g(s.x)) for s in mins), 1e-7)
z = np.linspace(-1.7, 1.7, 500)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(z, (z * z - 1) ** 2 + 0.2 * z)
for s in mins:
    ax.plot(s.x[0], s.fun, "o", label=f"국소 해 {s.x[0]:.3f}")
ax.set(xlabel="결정 변수", ylabel="목적함수", title="정상점과 전역 최소는 다름")
ax.legend()
savefig(fig, "nonconvex")

**결과 해석:** 두 시작점이 서로 다른 국소 최소에 도달합니다. 작은 기울기만으로 전역 최소를 식별할 수 없습니다.

#### 원본 예제 8.1 — 비선형 MPC의 두 국소 최소 (pp.495–496)

초기 상태 1, 예측 구간 1, 입력 범위 $[-1,1]$에서 원본 문제를 재현합니다.
$$x_1=1+u-2u^2,\qquad J=1+u^2+10x_1^2.$$
상태와 입력에 대한 비용은 볼록한 이차식이지만 동역학 등식의 실행 가능 집합은 곡선입니다. 상태를 대입하면
$$\phi(u)=11+20u-29u^2-40u^3+40u^4.$$
두 초기값에서 SLSQP를 실행하고, **도함수 다항식의 모든 실근과 구간 양 끝점**을 비교하여 이 1차원 문제의 전역 최소를 별도로 확인합니다. 곡선 위 두 점의 중점이 곡선 밖에 있으면 실행 가능 집합은 볼록하지 않습니다.

In [ ]:
def textbook_nonconvex():
    def state(u):
        return 1 + u - 2 * u * u

    def cost(u):
        return 1 + u * u + 10 * state(u) ** 2

    def derivative(u):
        return 2 * u + 20 * state(u) * (1 - 4 * u)

    roots = np.roots([160.0, -120.0, -58.0, 20.0])
    stationary = np.sort(roots.real[(abs(roots.imag) < 1e-12) & (abs(roots.real) < 1)])
    candidates = np.r_[-1.0, stationary, 1.0]
    best = candidates[np.argmin(cost(candidates))]
    solutions = []
    for initial in [-0.8, 0.8]:
        sol = minimize(
            lambda z: cost(z[0]),
            [initial],
            jac=lambda z: np.array([derivative(z[0])]),
            bounds=[(-1, 1)],
            method="SLSQP",
            options={"ftol": 1e-13, "maxiter": 100},
        )
        if not sol.success:
            raise RuntimeError(sol.message)
        solutions.append(sol.x[0])
    record(
        "원본 8.1 두 국소해 정지 잔차", max(abs(derivative(np.array(solutions)))), 2e-5
    )
    record("원본 8.1 전역 최소 입력", best)
    record("원본 8.1 전역 최소 비용", cost(best))
    assert cost(solutions[0]) < cost(solutions[1]) and abs(solutions[0] - best) < 1e-6
    grid = np.linspace(-1, 1, 501)
    record(
        "원본 8.1 축약 다항식 일치",
        max(
            abs(
                cost(grid)
                - (11 + 20 * grid - 29 * grid**2 - 40 * grid**3 + 40 * grid**4)
            )
        ),
        1e-12,
    )
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.7))
    axs[0].plot(grid, state(grid), label="실행 가능 곡선")
    axs[0].plot([-0.5, 1], [state(-0.5), state(1)], "--", label="두 점의 연결 선분")
    axs[0].set(xlabel="입력 u", ylabel="다음 상태", title="동역학 등식은 비볼록 집합")
    axs[0].legend()
    axs[1].semilogy(grid, cost(grid), label="축약 비용")
    for u, label in zip(solutions, ["전역 최소", "다른 국소 최소"]):
        axs[1].plot(u, cost(u), "o", label=label)
    axs[1].set(xlabel="입력 u", ylabel="비용 (로그 축)", title="초기값에 따른 국소 해")
    axs[1].legend()
    print("국소 입력:", solutions, "전역 최소 비용:", cost(best))
    savefig(fig, "textbook01")


textbook_nonconvex()

### 8.1.3 Simultaneous Versus Sequential Optimal Control — 동시·순차 최적 제어

**교재 인쇄 pp.496–498**

순차 접근은 입력에서 상태를 전진 모사합니다. 동시 접근은 상태를 변수로 남기고 결함을 제약합니다. 변수 수와 희소 구조를 함께 비교해야 합니다.

두 표현은 같은 정확한 이산 문제를 나타낼 수 있지만 최적화 반복은 같지 않습니다. 순차 접근은 상태를 항상 입력으로 재생성하지만 도함수가 조밀해질 수 있습니다. 동시 접근은 중간 반복에서 동역학 결함을 허용하고 국소 결합의 희소성을 유지합니다. 비선형 문제에서는 다른 국소 해에 수렴할 수도 있습니다.

### 8.1.4 Continuous Time Optimal Control Problem — 연속 시간 최적 제어 문제

**교재 인쇄 pp.498–501**

연속 궤적의 최적화는 무한 차원 문제입니다. 직접 방법은 먼저 유한 차원으로 이산화합니다. 상태 적분과 비용 구적의 정확도를 함께 봅니다.

입력을 구간마다 상수로 두어도 상태는 구간 안에서 움직입니다. 격자점에서 제약을 만족한다고 구간 전체에서 자동으로 만족하는 것은 아닙니다. §8.5에서 RK4 상태 적분과 같은 RK4 구적을 사용하고, 중점 배치법은 중점 구적과 함께 별도 이산 문제로 비교합니다.

$$
J=\int_0^T\ell_c(x(t),u(t))\,dt+V_f(x(T)),\qquad \dot x=f_c(x,u)
$$

## 8.2 Numerical Simulation — 수치 시뮬레이션

**교재 인쇄 pp.501–502**

연속 모델을 한 단계 상태 전이로 만듭니다. 적분 간격과 방법이 정확도·안정성을 바꿉니다. 고정 간격과 적응 간격의 용도를 구분합니다.

최적화 솔버가 정확히 푸는 대상은 실제 연속 시스템 자체가 아니라 구현한 이산 모델입니다. 적분 오차와 최적화 종료 오차를 따로 확인해야 합니다. 이 절은 알려진 정확한 해 또는 독립 고정밀 적분 결과를 기준으로 수치 방법을 비교합니다.

### 8.2.1 Explicit Runge-Kutta Methods — 명시적 Runge–Kutta 방법

**교재 인쇄 pp.502–506**

각 단계의 기울기를 순서대로 평가합니다. 방법의 차수는 간격 축소에 따른 오차율입니다. 더 높은 차수도 임의의 큰 간격을 허용하지 않습니다.

시험 방정식은 ẋ=−x, x(0)=1이고 t=1의 정확한 해는 exp(−1)입니다. 간격 h를 절반으로 줄일 때 전역 오차가 각각 약 2, 4, 16배 감소하는지 검사합니다. 매우 작은 간격에서는 반올림 오차 때문에 이 비율이 흐려질 수 있습니다.

$$
k_1=f(x),\ k_2=f(x+hk_1/2),\ k_3=f(x+hk_2/2),\ k_4=f(x+hk_3),\quad x^+=x+\frac h6(k_1+2k_2+2k_3+k_4)
$$

In [ ]:
def rk4(f, x, h):
    k1 = f(x)
    k2 = f(x + h * k1 / 2)
    k3 = f(x + h * k2 / 2)
    k4 = f(x + h * k3)
    return x + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6


ns = 2 ** np.arange(2, 9)
hs = 1 / ns
errors = {}
for name, step in [
    ("오일러", lambda x, h: x - h * x),
    ("중점 RK2", lambda x, h: x - h * (x - h * x / 2)),
    ("RK4", lambda x, h: rk4(lambda z: -z, x, h)),
]:
    ee = []
    for n in ns:
        x = 1.0
        for k in range(n):
            x = step(x, 1 / n)
        ee.append(abs(x - np.exp(-1)))
    errors[name] = np.array(ee)
orders = {name: np.log2(v[-3] / v[-2]) for name, v in errors.items()}
for name, p in zip(errors, (1, 2, 4)):
    record(name + " 관측 차수 차이", abs(orders[name] - p), 0.15)
fig, ax = plt.subplots(figsize=(8, 3.7))
for name, v in errors.items():
    ax.loglog(hs, v, "o-", label=f"{name} (관측 차수 {orders[name]:.2f})")
ax.set(
    xlabel="적분 간격 h", ylabel="t=1의 전역 오차", title="ẋ=−x, x(0)=1: 간격과 정확도"
)
ax.legend()
savefig(fig, "integration_order")

**결과 해석:** 해석적 해와 비교한 전역 오차에서 오일러·RK2·RK4의 1·2·4차 수렴을 확인합니다.

#### 원본 예제 8.3 — 회전계와 함수 평가 수 (pp.504–505)

$$\dot x=Ax,\quad A=\begin{bmatrix}0&1\\-1&0\end{bmatrix},\quad x(0)=(1,0),\quad T=2\pi.$$
정확한 해는 $(\cos t,-\sin t)$입니다. 교재의 **오일러·Heun·RK4**를 구현합니다. 앞선 RK2 중점법과 여기의 Heun은 서로 다른 방법이지만 모두 2차입니다.
Heun은 예측 $k_2=f(x+hf(x))$ 후 두 기울기의 평균으로 갱신합니다. 단계 수가 $s$이면 함수 평가 수는 $M=sN$입니다. 동일한 간격에서의 비교와 동일한 계산량에서의 비교를 구분하세요.

원본 문제와 방법을 재현하되 $N=16,\ldots,4096$의 제한된 격자를 사용합니다. 교재 그림의 전체 계산량 범위나 반올림 오차 바닥까지 재현하는 것은 아닙니다. 왼쪽 기울기는 마지막 네 격자의 로그 회귀로 추정합니다.

In [ ]:
def textbook_rotation():
    matrix = np.array([[0.0, 1.0], [-1.0, 0.0]])

    def step(x, h, method):
        first = matrix @ x
        if method == "오일러":
            return x + h * first
        if method == "Heun":
            return x + h * (first + matrix @ (x + h * first)) / 2
        return rk4(lambda y: matrix @ y, x, h)

    counts = 2 ** np.arange(4, 13)
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.7))
    for name, stages, order in [("오일러", 1, 1), ("Heun", 2, 2), ("RK4", 4, 4)]:
        errors = []
        for count in counts:
            value = np.array([1.0, 0.0])
            h = 2 * np.pi / count
            for _ in range(count):
                value = step(value, h, name)
            errors.append(np.linalg.norm(value - [1, 0]))
        observed = -np.polyfit(np.log(counts[-4:]), np.log(errors[-4:]), 1)[0]
        record(f"원본 8.3 {name} 관측 차수", observed)
        assert abs(observed - order) < 0.04
        axs[0].loglog(stages * counts, errors, "o-", label=f"{name}: {observed:.2f}차")
        # 같은 총 함수 평가 수 M=80에서 궤적을 비교합니다.
        count = 80 // stages
        value = np.array([1.0, 0.0])
        path = [value.copy()]
        for _ in range(count):
            value = step(value, 2 * np.pi / count, name)
            path.append(value.copy())
        path = np.array(path)
        axs[1].plot(path[:, 0], path[:, 1], label=name)
    theta = np.linspace(0, 2 * np.pi, 401)
    axs[1].plot(np.cos(theta), -np.sin(theta), "k--", label="정확한 원")
    axs[0].set(
        xlabel="함수 평가 수 M=sN", ylabel="종단 상태 오차", title="차수와 계산량"
    )
    axs[0].legend()
    axs[1].set(xlabel="상태 1", ylabel="상태 2", title="동일 계산량 M=80")
    axs[1].axis("equal")
    axs[1].legend()
    savefig(fig, "textbook03")


textbook_rotation()

### 8.2.2 Stiff Equations and Implicit Integrators — 강직 방정식과 음함수 적분기

**교재 인쇄 pp.506–507**

빠른 감쇠 모드가 명시적 적분 간격을 제한합니다. 음함수 단계는 다음 상태를 방정식으로 구합니다. 수치 안정성과 높은 정확도는 다른 성질입니다.

보충 예제 λ=−50, h=0.05에서 명시적 오일러의 증폭 인자는 −1.5여서 해가 교대로 커집니다. 음함수 오일러는 감쇠하지만 정확한 지수 함수와는 차이가 납니다. 명시적 오일러의 실수 음의 λ에 대한 안정 조건은 |1+hλ|<1입니다.

In [ ]:
# 강직 보충계: 빠른 감쇠 모드 −50. 오일러는 h<0.04에서만 안정.
h = 0.05
n = 20
t = np.arange(n + 1) * h
explicit = (-1.5) ** np.arange(n + 1)
implicit = (1 / 3.5) ** np.arange(n + 1)
assert abs(explicit[-1]) > 100 and abs(implicit[-1]) < 1e-8
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(t, explicit, "o-", label="명시적 오일러")
axs[1].plot(t, np.exp(-50 * t), label="정확한 해")
axs[1].plot(t, implicit, "o-", label="음함수 오일러")
for ax in axs:
    ax.set_xlabel("시간")
    ax.set_ylabel("상태")
    ax.legend()
axs[0].set_title("같은 간격에서 수치 불안정")
axs[1].set_title("안정성과 정확도는 별개")
savefig(fig, "stiff")

**결과 해석:** 명시적 오일러는 실제로 감쇠하는 계에서도 발산합니다. 음함수 오일러의 안정성이 정확한 궤적 일치를 뜻하지는 않습니다.

### 8.2.3 Implicit Runge-Kutta and Collocation Methods — 음함수 RK와 배치법

**교재 인쇄 pp.507–511**

내부 단계들이 서로 연결된 방정식을 풉니다. 배치점에서 다항식의 미분과 동역학을 맞춥니다. A-안정성과 강한 감쇠 특성을 구분합니다.

배치법은 한 구간의 상태를 다항식으로 나타내고 지정한 배치점에서 미분 방정식을 만족시킵니다. 1점 Gauss 배치는 음함수 중점법이며 2차 정확도를 가집니다. A-안정이지만 매우 큰 음의 hλ에서 증폭 인자가 −1로 접근하므로 강한 감쇠를 제거하는 L-안정과는 다릅니다.

$$
k_i=f\!\left(x+h\sum_j a_{ij}k_j\right),\qquad x^+=x+h\sum_i b_i k_i
$$

In [ ]:
# 1점 Gauss 배치법 = 음함수 중점법. 여기서는 선형 시험 방정식의 해석적 단계.
ns2 = 2 ** np.arange(2, 8)
ee = []
for n in ns2:
    h = 1 / n
    approx = ((1 - h / 2) / (1 + h / 2)) ** n
    ee.append(abs(approx - np.exp(-1)))
record("음함수 중점법 관측 차수 차이", abs(np.log2(ee[-2] / ee[-1]) - 2), 0.02)
# 강한 감쇠에서 중점법의 증폭 인자는 −1에 가까워진다: A-안정 ≠ L-안정.
z = -np.logspace(-2, 3, 300)
amp_mid = (1 + z / 2) / (1 - z / 2)
amp_ie = 1 / (1 - z)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.semilogx(-z, abs(amp_mid), label="음함수 중점")
ax.semilogx(-z, abs(amp_ie), label="음함수 오일러")
ax.set(
    xlabel="감쇠 크기 |hλ|",
    ylabel="증폭 인자의 절댓값",
    title="A-안정 방법끼리도 강한 감쇠 처리는 다름",
)
ax.legend()
savefig(fig, "implicit_stability")

**결과 해석:** 두 방법 모두 음의 실수 시험 모드에 안정적이지만, 중점법은 아주 빠른 모드를 강하게 감쇠시키지 못합니다.

#### 원본 예제 8.4 — 강직한 비선형 회전계 (pp.511–512)

$$\dot x=Ax-500x(\|x\|^2-1),\qquad x(0)=(1,0),\quad T=2\pi.$$
$A$는 예제 8.3과 같습니다. 반경 오차가 빠르게 되돌아오는 반면 각도는 속도 1로 움직입니다. 정확한 해는 여전히 단위원입니다. 음함수 오일러, 1점 Gauss(GL2), 2점 Gauss(GL4)의 **단계 상태를 비선형 연립방정식으로 실제로 풉니다**.
$$Y_i=x+h\sum_j a_{ij}f(Y_j),\quad x^+=x+h\sum_i b_if(Y_i).$$
GL4의 대각 계수는 $1/4$, 비대각은 $1/4\mp\sqrt3/6$, 가중치는 각각 $1/2$입니다. 단계 잔차와 정확한 종단 해의 차이를 분리합니다. 배치점 수 $M=sN$은 교재의 비용 지표이며 실제 뉴턴 반복 횟수나 실행 시간과 같지 않습니다.

$N=32,64,128,256,512$를 사용합니다. 원본 그림의 $M=10$ 대간격 궤적이나 전체 오차 범위는 재현하지 않습니다. 이 실습에서 GL2의 노드 반경 진동과 위상 오차도 관찰합니다.

In [ ]:
def textbook_stiff():
    rotation = np.array([[0.0, 1.0], [-1.0, 0.0]])

    def fun(x):
        return rotation @ x - 500 * x * (x @ x - 1)

    def jac(x):
        return rotation - 500 * ((x @ x - 1) * np.eye(2) + 2 * np.outer(x, x))

    q = np.sqrt(3) / 6
    tableaus = [
        ("음함수 오일러", np.array([[1.0]]), np.array([1.0])),
        ("GL2", np.array([[0.5]]), np.array([1.0])),
        ("GL4", np.array([[0.25, 0.25 - q], [0.25 + q, 0.25]]), np.array([0.5, 0.5])),
    ]
    counts = np.array([32, 64, 128, 256, 512])
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.7))
    worst = 0.0
    for name, coeff, weights in tableaus:
        stages = len(weights)
        errors = []
        for count in counts:
            h = 2 * np.pi / count
            x = np.array([1.0, 0.0])
            path = [x.copy()]
            for _ in range(count):

                def residual(flat):
                    values = flat.reshape(stages, 2)
                    ff = np.array([fun(y) for y in values])
                    return (values - x - h * coeff @ ff).ravel()

                def jacobian(flat):
                    values = flat.reshape(stages, 2)
                    return np.block(
                        [
                            [
                                (np.eye(2) if i == j else np.zeros((2, 2)))
                                - h * coeff[i, j] * jac(values[j])
                                for j in range(stages)
                            ]
                            for i in range(stages)
                        ]
                    )

                # 현재 상태를 모든 단계 상태의 초기 추정값으로 사용합니다.
                sol = root(residual, np.tile(x, stages), jac=jacobian, tol=1e-10)
                err = np.max(abs(residual(sol.x)))
                worst = max(worst, err)
                if not np.all(np.isfinite(sol.x)) or err > 1e-9:
                    raise RuntimeError((name, count, sol.message, err))
                values = sol.x.reshape(stages, 2)
                x = x + h * weights @ np.array([fun(y) for y in values])
                path.append(x.copy())
            errors.append(np.linalg.norm(x - [1, 0]))
            if count == 32:
                radius = np.linalg.norm(path, axis=1) - 1
                axs[1].plot(np.linspace(0, 2 * np.pi, count + 1), radius, label=name)
        record(f"원본 8.4 {name} N512 종단 오차", errors[-1])
        assert errors[-1] < errors[0]
        axs[0].loglog(stages * counts, errors, "o-", label=name)
    record("원본 8.4 최대 단계 잔차", worst, 1e-9)
    axs[0].set(
        xlabel="총 배치점 수 M=sN", ylabel="종단 상태 오차", title="강직계의 적분 오차"
    )
    axs[0].legend()
    axs[1].set(xlabel="시간", ylabel="노드 반경 − 1", title="같은 N=32에서 반경 오차")
    axs[1].legend()
    savefig(fig, "textbook04")


textbook_stiff()

### 8.2.4 Differential Algebraic Equations — 미분 대수 방정식

**교재 인쇄 pp.511–513**

미분 상태와 대수 변수를 함께 계산합니다. 초기값부터 대수 제약을 만족해야 합니다. 이 예제의 대수 야코비안은 가역입니다.

보충 모델은 대수 변수 z를 제거할 수 있는 index-1 DAE입니다. 다음 상태와 대수 변수를 음함수 오일러 잔차 두 식으로 풀고, z=x²를 대입한 독립 ODE의 해와 비교합니다. 모든 DAE가 이런 방식으로 쉽게 제거되거나 동일한 초기화 조건을 가지는 것은 아닙니다.

$$
x^+-x-h(-x^++z^+)=0,\qquad z^+-(x^+)^2=0
$$

In [ ]:
# index-1 DAE: ẋ=−x+z, 0=z−x². 대수 변수를 제거한 독립 ODE와 비교.
h = 0.02
tt = np.arange(51) * h
state = [0.2]
alg = [0.04]
residual = []
for k in range(50):
    old = state[-1]
    fun = lambda v: np.array([v[0] - old - h * (-v[0] + v[1]), v[1] - v[0] ** 2])
    sol = root(fun, [old, old**2])
    assert sol.success
    residual.append(np.max(abs(fun(sol.x))))
    state.append(sol.x[0])
    alg.append(sol.x[1])
ref = solve_ivp(
    lambda t, x: -x + x * x, [0, 1], [0.2], rtol=1e-11, atol=1e-13, t_eval=tt
)
record("DAE 대수 제약 잔차", np.max(abs(np.array(alg) - np.array(state) ** 2)), 1e-9)
record("DAE 단계 방정식 잔차", max(residual), 1e-9)
record("DAE 이산화 오차", np.max(abs(np.array(state) - ref.y[0])), 0.003)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(tt, state, label="DAE 음함수 오일러")
ax.plot(tt, ref.y[0], "--", label="제거한 ODE 기준 해")
ax.plot(tt, alg, label="대수 변수 z=x²")
ax.set(xlabel="시간", ylabel="변수", title="일관된 초기값과 대수 제약")
ax.legend()
savefig(fig, "dae")

**결과 해석:** 대수 잔차와 단계 방정식 잔차를 검사하고, 독립 ODE와의 차이는 이산화 오차로 따로 확인합니다.

### 8.2.5 Integrator Adaptivity — 적응형 적분

**교재 인쇄 pp.513**

오차 추정에 따라 적분 간격을 조절합니다. 요청 허용오차는 실제 전역 오차 그 자체가 아닙니다. 정확도를 높이면 함수 평가가 늘 수 있습니다.

정확한 해가 cos(t)인 시험식 ẋ=−20(x−cos(t))−sin(t)를 사용합니다. 상대 허용오차 1e−3, 1e−6, 1e−9에서 실제 오차와 함수 평가 수를 기록합니다. 이 예제의 숫자는 보편적인 적분기 성능 순위가 아닙니다.

In [ ]:
# 알려진 정확한 해 x=cos(t), 빠른 과도응답 항을 포함한 시험식.
results = []
for tol in [1e-3, 1e-6, 1e-9]:
    sol = solve_ivp(
        lambda t, x: -20 * (x - np.cos(t)) - np.sin(t),
        [0, 3],
        [1.0],
        rtol=tol,
        atol=tol * 0.01,
        dense_output=True,
    )
    assert sol.success
    tt = np.linspace(0, 3, 1001)
    err = np.max(abs(sol.sol(tt)[0] - np.cos(tt)))
    results.append((tol, err, sol.nfev, len(sol.t) - 1))
record("적응 적분 엄격 허용오차의 전역 오차", results[-1][1], 1e-7)
assert results[-1][1] < results[0][1]
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
labels = [str(r[0]) for r in results]
axs[0].bar(labels, [r[1] for r in results])
axs[0].set_yscale("log")
axs[0].set(
    ylabel="관측 전역 오차", xlabel="요청 상대 허용오차", title="허용오차와 실제 오차"
)
axs[1].bar(labels, [r[2] for r in results])
axs[1].set(
    ylabel="함수 평가 수", xlabel="요청 상대 허용오차", title="이 문제에서의 계산량"
)
savefig(fig, "adaptivity")

**결과 해석:** 요청 허용오차와 관측된 전역 오차를 구별합니다. 오른쪽은 이 특정 문제에서의 함수 평가 수입니다.

## 8.3 Solving Nonlinear Equation Systems — 비선형 연립방정식 풀기

**교재 인쇄 pp.513**

음함수 적분과 KKT는 방정식 풀이로 연결됩니다. 뉴턴 단계마다 선형 시스템을 풉니다. 잔차와 단계 크기, 특이성을 함께 확인합니다.

최적화와 적분에서 반복적으로 나타나는 계산 단위는 잔차 함수와 야코비안입니다. 정확한 야코비안, 고정 또는 근사 야코비안의 선택은 단계 비용과 수렴률을 바꿉니다. 방정식의 잔차가 작은지 직접 검사합니다.

### 8.3.1 Linear Systems — 선형 시스템

**교재 인쇄 pp.513–514**

역행렬을 명시적으로 만들기보다 방정식을 풉니다. 조건수가 크면 작은 입력 오차가 증폭됩니다. 작은 잔차만으로 작은 해 오차를 단정하지 않습니다.

2×2 양의 정부호 시스템의 상대 잔차를 검사합니다. 이어서 대각 행렬 diag(1,1e−8)에 작은 우변 변화를 주어 해가 더 크게 바뀌는 모습을 확인합니다. 실제 계산은 선형 분해와 solve를 사용하며 역행렬을 직접 곱하지 않습니다.

In [ ]:
M = np.array([[4.0, 1.0], [1.0, 3.0]])
rhs = np.array([1.0, 2.0])
sol = np.linalg.solve(M, rhs)
record(
    "선형 시스템 상대 잔차", np.linalg.norm(M @ sol - rhs) / np.linalg.norm(rhs), 1e-12
)
# 작은 잔차가 작은 전진 오차를 보장하지 않는 조건수 예.
ill = np.diag([1.0, 1e-8])
true = np.ones(2)
rr = ill @ true
pert = rr + np.array([0.0, 1e-10])
changed = np.linalg.solve(ill, pert)
print(
    "조건수:",
    np.linalg.cond(ill),
    "상대 우변 변화:",
    np.linalg.norm(pert - rr) / np.linalg.norm(rr),
    "해 변화:",
    np.linalg.norm(changed - true),
)

### 8.3.2 Nonlinear Root-Finding Problems — 비선형 근 찾기

**교재 인쇄 pp.514–517**

현재 점에서 잔차를 선형화합니다. 선형 시스템의 해를 보정량으로 사용합니다. 근처의 정칙한 근에서 빠른 수렴을 기대합니다.

제곱근 문제를 시작점 2에서 풀고 마지막 잔차를 검사합니다. 코드에는 작은 잔차 종료와 작은 도함수 검사를 둡니다. 도함수가 너무 작으면 예외를 발생시킵니다. 반복 한도에 도달한 이력은 수렴을 의미하지 않으므로 사용 지점에서 최종 잔차를 확인합니다.

$$
J(z_k)\Delta z_k=-R(z_k),\qquad z_{k+1}=z_k+\Delta z_k
$$

In [ ]:
def newton_scalar(f, df, x, iterations=12, damped=False):
    x = finite_scalar(x, "초기값")
    iterations = positive_integer(iterations, "반복 수")
    hist = [float(x)]
    alphas = []
    for i in range(iterations):
        r = finite_scalar(f(x), "잔차")
        j = finite_scalar(df(x), "도함수")
        if abs(r) < 1e-13:
            break
        if abs(j) < 1e-14:
            raise RuntimeError("도함수가 작아 뉴턴 단계를 계산할 수 없습니다.")
        step = -r / j
        alpha = 1.0
        if damped:
            while 0.5 * f(x + alpha * step) ** 2 > (1 - 1e-4 * alpha) * 0.5 * r * r:
                alpha *= 0.5
                if alpha < 1e-12:
                    raise RuntimeError("감쇠 탐색 실패")
        x += alpha * step
        hist.append(float(x))
        alphas.append(alpha)
    return np.array(hist), np.array(alphas)


n_hist, _ = newton_scalar(lambda z: z * z - 2, lambda z: 2 * z, 2.0)
record("뉴턴 제곱근 최종 잔차", abs(n_hist[-1] ** 2 - 2), 1e-12)

### 8.3.3 Local Convergence of Newton-Type Methods — 뉴턴형 방법의 국소 수렴

**교재 인쇄 pp.517–519**

정확한 뉴턴은 정칙성 아래 국소 이차 수렴합니다. 고정 야코비안은 보통 더 느리게 수렴합니다. 수렴률은 시작점과 매끄러움 조건에 의존합니다.

같은 z²−2 문제에서 정확한 뉴턴과 도함수를 4로 고정한 반복을 비교합니다. 그래프는 정확한 근과의 거리이고 1e−16 아래는 표시를 위해 바닥값을 둡니다. 선형·이차 수렴 주장은 해당 근 근처의 국소 성질입니다.

$$
\|e_{k+1}\|\le C\|e_k\|^2\quad\text{(local Newton convergence)}
$$

In [ ]:
fixed = [2.0]
for i in range(35):
    fixed.append(fixed[-1] - (fixed[-1] ** 2 - 2) / 4)
en = abs(n_hist - np.sqrt(2))
ef = abs(np.array(fixed) - np.sqrt(2))
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.semilogy(np.maximum(en, 1e-16), "o-", label="정확한 뉴턴")
ax.semilogy(np.maximum(ef, 1e-16), ".-", label="고정 야코비안")
ax.set(xlabel="반복 횟수", ylabel="근과의 거리", title="국소 이차 수렴과 선형 수렴")
ax.legend()
savefig(fig, "newton_convergence")
record("고정 야코비안 최종 오차", ef[-1], 1e-10)

**결과 해석:** 같은 근에 수렴해도 오차 감소 속도가 다릅니다. 기계 정밀도에 도달한 뒤의 표시값은 실제 수렴률 분석에서 제외합니다.

#### 원본 예제 8.5 — 다섯제곱근의 뉴턴 반복 (pp.516–517)

원본의 $R(z)=z^5-2$, $z_0=2$를 그대로 사용합니다. 정확한 뉴턴은 매번 $5z_k^4$를 계산하고, 고정 야코비안 방법은 처음 값 80을 재사용합니다. 첫 반복은 둘 다 $13/8$입니다.

근 $z^*=2^{1/5}$ 주변에서 고정 방법의 선형 수렴 인자는 $|1-5(z^*)^4/80|$입니다. 정확한 뉴턴의 이차 오차 비율은 $|e_{k+1}|/|e_k|^2\to 2/z^*$입니다. 반올림 수준의 오차를 이 비율에 넣지 않습니다. 뒤의 원본 예제 8.6이 분류하는 선형·이차 수렴을 실제 근 찾기에 연결합니다.

In [ ]:
def textbook_newton():
    exact = 2**0.2
    nh, _ = newton_scalar(lambda z: z**5 - 2, lambda z: 5 * z**4, 2.0, iterations=20)
    frozen = [2.0]
    for _ in range(180):
        frozen.append(frozen[-1] - (frozen[-1] ** 5 - 2) / 80)
    frozen = np.array(frozen)
    ne = abs(nh - exact)
    fe = abs(frozen - exact)
    ratio = fe[121] / fe[120]
    predicted = abs(1 - 5 * exact**4 / 80)
    record("원본 8.5 첫 반복 오차", abs(nh[1] - 13 / 8), 1e-14)
    record("원본 8.5 뉴턴 최종 잔차", abs(nh[-1] ** 5 - 2), 1e-12)
    record("원본 8.5 고정 야코비안 수렴 인자", predicted)
    record("원본 8.5 선형 수렴 비율 차이", abs(ratio - predicted), 1e-6)
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.7))
    axs[0].semilogy(
        np.arange(len(ne))[ne > 1e-15], ne[ne > 1e-15], "o-", label="정확한 뉴턴"
    )
    axs[0].semilogy(fe, label="고정 야코비안 80")
    axs[0].set(
        xlabel="반복 k", ylabel="근의 절대 오차", title="첫 단계는 같아도 수렴은 다름"
    )
    axs[0].legend()
    valid = (ne[:-1] > 1e-6) & (ne[1:] > 1e-14)
    axs[1].plot(
        np.arange(len(ne) - 1)[valid],
        (ne[1:] / ne[:-1] ** 2)[valid],
        "o-",
        label="관측 이차 오차 비율",
    )
    axs[1].axhline(2 / exact, ls="--", label="국소 극한 2/z*")
    axs[1].set(
        xlabel="뉴턴 반복 k", ylabel="다음 오차 / 현재 오차²", title="국소 이차 수렴"
    )
    axs[1].legend()
    savefig(fig, "textbook05")


textbook_newton()

### 8.3.4 Affine Invariance — 아핀 불변성

**교재 인쇄 pp.519**

가역 잔차 선형 변환은 정확한 뉴턴 단계를 보존합니다. 근사 풀이와 유한 정밀도에서는 차이가 생길 수 있습니다. 잔차 노름 기반 종료와 선 탐색은 스케일에 민감합니다.

보충 실습은 2변수 비선형 잔차에 가역 행렬 T를 곱한 전후의 선형 시스템 해를 비교합니다. 이는 정확한 전체 뉴턴 단계에 대한 비교입니다. 잔차 노름을 기준으로 한 감쇠와 종료 규칙까지 자동으로 동일해진다고 해석하지 않습니다.

$$
(TJ)\Delta z=-TR\quad\Longleftrightarrow\quad J\Delta z=-R\qquad (T\text{ invertible})
$$

In [ ]:
# 가역인 잔차 선형 변환은 정확한 전체 뉴턴 단계를 바꾸지 않는다.
fun = lambda z: np.array([z[0] ** 2 + z[1] - 2, z[0] + z[1] ** 2 - 2])
jac = lambda z: np.array([[2 * z[0], 1.0], [1.0, 2 * z[1]]])
z = np.array([1.4, 0.8])
T = np.array([[10.0, 2.0], [0.0, 0.1]])
d1 = np.linalg.solve(jac(z), -fun(z))
d2 = np.linalg.solve(T @ jac(z), -T @ fun(z))
record("잔차 선형 변환 뉴턴 단계 차이", np.max(abs(d1 - d2)), 1e-12)

### 8.3.5 Globalization for Newton-Type Methods — 뉴턴형 방법의 수렴 영역 확장

**교재 인쇄 pp.519–520**

전체 뉴턴 단계는 멀리서 순환하거나 발산할 수 있습니다. 메리트 함수 감소로 단계 길이를 조절할 수 있습니다. 감쇠도 모든 시작점에서 근을 보장하지 않습니다.

전체 단계는 시작점 0에서 0과 1을 오갑니다. 감쇠 실습은 별도 시작점 −0.5를 사용하며 초기값 차이를 그래프에 표시합니다. 잔차 제곱에는 잔차가 0이 아닌 정상점도 있을 수 있으므로, 감소 검사만으로 근 발견을 주장하지 않고 최종 잔차를 검사합니다.

$$
\psi(z)=\frac12\|R(z)\|^2,\qquad z^+=z+\alpha\Delta z
$$

In [ ]:
f = lambda z: z**3 - 2 * z + 2
df = lambda z: 3 * z * z - 2
full, _ = newton_scalar(f, df, 0.0, iterations=16)
# 같은 시작 0에서 단순 잔차 탐색이 정상점에 걸릴 수도 있으므로,
# 별도 시작 −0.5에서 감쇠의 효과를 검증하고 전체 단계와 비교한다.
damp, alphas = newton_scalar(f, df, -0.5, iterations=40, damped=True)
record("감쇠 뉴턴 최종 잔차", abs(f(damp[-1])), 1e-10)
assert abs(f(full[-1])) > 1
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(full, "o-", label="전체 단계, 시작 0")
axs[0].plot(damp, "o-", label="감쇠 단계, 시작 −0.5")
axs[0].set(xlabel="반복", ylabel="변수", title="시작점과 단계 선택을 함께 명시")
axs[0].legend()
axs[1].semilogy(np.maximum(abs(f(damp)), 1e-16), "o-")
axs[1].set(xlabel="반복", ylabel="잔차", title="감쇠는 일반적인 전역 해 보장이 아님")
savefig(fig, "globalization")

**결과 해석:** 전체 단계의 순환과 감쇠 반복의 수렴을 보여 줍니다. 시작점도 함께 명시하며 감쇠의 보편적 성공을 주장하지 않습니다.

## 8.4 Computing Derivatives — 도함수 계산

**교재 인쇄 pp.520–521**

최적화와 감도 계산에는 정확한 도함수가 중요합니다. 차분과 알고리즘 미분의 오차 원인을 구분합니다. 필요한 전체 행렬 또는 방향 곱을 선택합니다.

최적화 반복에서 비용 기울기와 제약 야코비안은 핵심 입력입니다. 전체 야코비안을 구성할 필요 없이 방향 곱만으로 계산할 수 있는 경우가 많습니다. 도함수의 정확성을 독립 해석식이나 차분으로 점검합니다.

### 8.4.1 Numerical Differentiation — 수치 미분

**교재 인쇄 pp.521–522**

큰 차분 간격에서는 절단 오차가 큽니다. 작은 간격에서는 뺄셈 상쇄가 커집니다. 간격을 줄이기만 해서는 정확해지지 않습니다.

exp(x)의 x=0.7 도함수를 알려진 해와 비교합니다. 간격을 1e−1부터 1e−16까지 바꾸며 전진·중심 차분의 오차를 확인합니다. 한 간격의 수치 미분을 절대적인 정답으로 사용하지 않습니다.

$$
f'(x)\approx\frac{f(x+h)-f(x-h)}{2h}
$$

In [ ]:
x = 0.7
hs = np.logspace(-1, -16, 80)
truth = np.exp(x)
forward = abs((np.exp(x + hs) - np.exp(x)) / hs - truth)
central = abs((np.exp(x + hs) - np.exp(x - hs)) / (2 * hs) - truth)
record("중심 차분 최소 오차", np.min(central), 1e-8)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.loglog(hs, np.maximum(forward, 1e-17), label="전진 차분")
ax.loglog(hs, np.maximum(central, 1e-17), label="중심 차분")
ax.set(
    xlabel="차분 간격", ylabel="도함수 절대 오차", title="절단 오차와 부동소수점 상쇄"
)
ax.legend()
savefig(fig, "finite_difference")

**결과 해석:** 차분 간격을 지나치게 줄이면 정확도가 나빠집니다. 중심 차분도 부동소수점 상쇄를 피하지 못합니다.

### 8.4.2 Algorithmic Differentiation — 알고리즘 미분

**교재 인쇄 pp.522–523**

기본 연산 각각에 연쇄법칙을 적용합니다. 실제로 실행되는 계산 그래프를 미분합니다. 차분 간격에 따른 절단 오차가 없습니다.

보충 함수 y=sin(ab)+a²를 곱셈, 사인, 제곱, 합으로 나눕니다. 실수 연산의 반올림 오차는 남지만 차분 간격은 필요 없습니다. 분기나 비매끄러운 연산이 있는 함수에서는 선택한 경로의 미분과 원래 수학적 미분의 관계를 검토해야 합니다.

In [ ]:
# 직접 설계한 계산 그래프 y=sin(ab)+a²: 곱셈 노드를 여러 경로에서 사용.
def graph(a, b):
    v = a * b
    s = np.sin(v)
    q = a * a
    return s + q


point = np.array([0.7, -0.4])
direction = np.array([0.3, 0.8])
exact_grad = np.array(
    [
        np.cos(np.prod(point)) * point[1] + 2 * point[0],
        np.cos(np.prod(point)) * point[0],
    ]
)

### 8.4.3 Implicit Function Interpretation — 음함수 관점의 미분

**교재 인쇄 pp.523–526**

계산 그래프를 잔차 방정식으로 볼 수 있습니다. 음함수 미분은 선형 시스템의 해입니다. 잔차 야코비안의 정칙성이 필요합니다.

z+z³=p에서는 R_z=1+3z²가 항상 양수여서 dz/dp=1/(1+3z²)입니다. 선형 시스템 관점의 감도를 서로 다른 p에서 근을 다시 구한 중심 차분과 비교합니다. 특이한 야코비안에서는 이 공식을 그대로 사용할 수 없습니다.

$$
R_z\frac{\partial z^*}{\partial p}=-R_p
$$

In [ ]:
# 음함수 z+z³=p의 감도. R_z=1+3z²는 항상 양수.
p = 0.8
z = root_scalar(lambda z: z + z**3 - p, bracket=[0, 1], xtol=1e-14).root
sens = 1 / (1 + 3 * z * z)
eps = 1e-5
zs = [
    root_scalar(lambda z: z + z**3 - v, bracket=[0, 1], xtol=1e-14).root
    for v in (p - eps, p + eps)
]
record("음함수 감도와 차분 비교", abs(sens - (zs[1] - zs[0]) / (2 * eps)), 1e-8)
print("R_z * dz/dp + R_p =", (1 + 3 * z * z) * sens - 1)

### 8.4.4 Algorithmic Differentiation in Forward Mode — 전진 모드 알고리즘 미분

**교재 인쇄 pp.526–528**

입력 방향을 계산 그래프의 순서대로 전달합니다. 한 번의 전파로 야코비안 방향 곱을 얻습니다. 입력 방향 수가 적을 때 유용합니다.

점 (0.7,−0.4), 방향 (0.3,0.8)에서 전진 모드를 손으로 구현합니다. 해석적 기울기와 방향 벡터의 내적을 기준으로 확인합니다. 방향 점 표기는 실제 시간 미분과 구분해야 합니다.

$$
\delta y=\cos(ab)(b\,\delta a+a\,\delta b)+2a\,\delta a
$$

In [ ]:
a, b = point
da, db = direction
v = a * b
dv = da * b + a * db
ds = np.cos(v) * dv
dq = 2 * a * da
dy = ds + dq
record("전진 AD 방향 미분 차이", abs(dy - exact_grad @ direction), 1e-12)

### 8.4.5 Algorithmic Differentiation in Reverse Mode — 역방향 모드 알고리즘 미분

**교재 인쇄 pp.528–531**

출력 수반에서 입력 쪽으로 역순 전파합니다. 공유 노드로 들어오는 기여를 누적합니다. 스칼라 비용의 많은 입력 기울기에 유용합니다.

전진 모드와 같은 그래프를 역순으로 계산합니다. a는 곱셈 ab와 제곱 a²에 함께 쓰이므로 두 경로의 기여를 더해야 합니다. 이 작은 예제는 전체 기울기를 해석식과 비교합니다. 역방향 계산은 보통 중간값 저장 또는 재계산이 필요합니다.

$$
\nabla y=\begin{bmatrix}b\cos(ab)+2a\\a\cos(ab)\end{bmatrix}
$$

In [ ]:
# 출력 수반을 1로 시작해 모든 경로 기여를 += 로 누적.
bar_s = 1.0
bar_q = 1.0
bar_v = np.cos(v) * bar_s
bar_a = 2 * a * bar_q
bar_b = 0.0
bar_a += b * bar_v
bar_b += a * bar_v
reverse = np.array([bar_a, bar_b])
record("역방향 AD 기울기 차이", np.max(abs(reverse - exact_grad)), 1e-12)
fig, ax = plt.subplots(figsize=(8, 3.6))
positions = np.arange(2)
ax.bar(positions - 0.16, reverse, 0.32, label="역방향 누적")
ax.bar(positions + 0.16, exact_grad, 0.32, label="해석적 미분")
ax.set(
    xticks=positions,
    xticklabels=["입력 a", "입력 b"],
    ylabel="기울기",
    title="공유 노드의 기여를 합산",
)
ax.legend()
savefig(fig, "ad_modes")

**결과 해석:** 두 입력의 기울기가 해석적 도함수와 일치합니다. 공유된 입력에는 여러 연산 경로의 기여를 누적합니다.

#### 원본 예제 8.9·8.11·8.12 — 3입력·2출력 계산 그래프

**원본 pp.522–523, 526–527, 528–531.** 원본의 함수를 사용합니다.
$$F(u)=\begin{bmatrix}u_1u_2u_3\\\sin(u_1u_2)+\exp(u_1u_2u_3)\end{bmatrix}.$$
중간값 $a=u_1u_2$, $b=\sin a$, $c=au_3$, $d=\exp c$를 순서대로 계산합니다. 전진 모드는 입력 방향 $v$를 전달하여 $Jv$를, 역방향 모드는 출력 가중치 $w$를 전달하여 $J^Tw$를 구합니다. 출력 $c$는 첫 출력과 지수함수 경로에 동시에 기여하므로 역방향에서 **두 기여를 더합니다**.

원본 계산 그래프를 재현하고 검증점 $(0.7,-0.4,0.5)$ 및 고정 난수 표본은 보충 설정으로 선택했습니다. 독립 해석 야코비안, CasADi, 쌍대 항등식 $w^T(Jv)=v^T(J^Tw)$를 대조합니다. 전체 야코비안은 입력 기저 3회 또는 출력 기저 2회 전파로 얻습니다. 이것은 방향별 전파 횟수이며 전체 실행 시간의 비율을 뜻하지 않습니다.

In [ ]:
def textbook_forward(u, v):
    u = finite_vector(u, 3, "입력")
    v = finite_vector(v, 3, "방향")
    a = u[0] * u[1]
    da = v[0] * u[1] + u[0] * v[1]
    c = a * u[2]
    dc = da * u[2] + a * v[2]
    return np.array([dc, np.cos(a) * da + np.exp(c) * dc])


def textbook_reverse(u, w):
    u = finite_vector(u, 3, "입력")
    w = finite_vector(w, 2, "출력 가중치")
    a = u[0] * u[1]
    c = a * u[2]
    bar_c = w[0] + w[1] * np.exp(c)
    bar_a = w[1] * np.cos(a) + bar_c * u[2]
    return np.array([bar_a * u[1], bar_a * u[0], bar_c * a])


def textbook_ad():
    import casadi as ca

    def analytical(u):
        a = u[0] * u[1]
        c = a * u[2]
        first = np.array([u[1] * u[2], u[0] * u[2], a])
        return np.vstack(
            [first, np.cos(a) * np.array([u[1], u[0], 0]) + np.exp(c) * first]
        )

    symbol = ca.SX.sym("original", 3)
    product = symbol[0] * symbol[1]
    out = ca.vertcat(product * symbol[2], ca.sin(product) + ca.exp(product * symbol[2]))
    cf = ca.Function("original_ad", [symbol], [ca.jacobian(out, symbol)])
    rng = np.random.default_rng(808)
    samples = np.vstack([[0.7, -0.4, 0.5], rng.uniform(-0.8, 0.8, (20, 3))])
    errs = np.zeros(4)
    for u in samples:
        jf = np.column_stack([textbook_forward(u, v) for v in np.eye(3)])
        jr = np.vstack([textbook_reverse(u, w) for w in np.eye(2)])
        truth = analytical(u)
        v = rng.normal(size=3)
        w = rng.normal(size=2)
        errs = np.maximum(
            errs,
            [
                np.max(abs(jf - truth)),
                np.max(abs(jr - truth)),
                np.max(abs(np.array(cf(u)) - truth)),
                abs(w @ textbook_forward(u, v) - v @ textbook_reverse(u, w)),
            ],
        )
    for name, error in zip(["전진 모드", "역방향 모드", "CasADi", "쌍대 항등식"], errs):
        record(f"원본 AD {name} 오차", error, 1e-12)
    u = samples[0]
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.7))
    for ax, mat, title in zip(
        axs,
        [analytical(u), np.vstack([textbook_reverse(u, w) for w in np.eye(2)])],
        ["해석 야코비안", "역방향 2회로 만든 야코비안"],
    ):
        ax.imshow(mat, cmap="coolwarm", vmin=-1, vmax=1)
        ax.grid(False)
        for i in range(2):
            for j in range(3):
                ax.text(j, i, f"{mat[i,j]:.4f}", ha="center", va="center")
        ax.set(
            xticks=range(3),
            xticklabels=["u₁", "u₂", "u₃"],
            yticks=range(2),
            yticklabels=["출력 1", "출력 2"],
            title=title,
        )
    savefig(fig, "textbook_ad")


textbook_ad()

### 8.4.6 Differentiation of Simulation Routines — 시뮬레이션의 미분

**교재 인쇄 pp.531–533**

상태와 민감도 방정식을 함께 적분할 수 있습니다. 이산 계산의 미분과 연속 해의 미분을 구분합니다. 적응 분기와 반복 종료의 영향을 고려합니다.

ẋ=−px, x(0)=1에서 감도는 Ṡ=−x−pS, S(0)=0입니다. 상태와 감도를 함께 RK4로 적분하여 연속 해의 감도 −exp(−p)와 비교합니다. 이 차이에는 이산화 오차가 포함됩니다. 적응 적분 코드의 분기 경로 미분과 정확한 흐름의 감도는 주의해서 연결해야 합니다.

$$
\dot S=f_xS+f_p,\qquad S(0)=\frac{\partial x(0)}{\partial p}
$$

In [ ]:
# ẋ=−p x 의 상태와 파라미터 감도를 같은 RK4로 적분한다.
p = 0.8
h = 0.05
aug = np.array([1.0, 0.0])
for k in range(20):
    aug = rk4(lambda y: np.array([-p * y[0], -y[0] - p * y[1]]), aug, h)
record("적분 감도와 연속 해 차이", abs(aug[1] + np.exp(-p)), 1e-7)
# 고정된 이산 계산의 미분과 연속 해의 미분은 이산화 오차만큼 다를 수 있다.

### 8.4.7 Algorithmic and Symbolic Differentiation Software — 알고리즘·기호 미분 도구

**교재 인쇄 pp.533**

모델을 도구가 이해하는 계산 그래프로 표현합니다. 기호식 조작과 알고리즘 미분을 구분합니다. 생성한 도함수를 작은 독립 예제로 확인합니다.

CasADi의 SX 식과 Function을 사용해 같은 y=sin(ab)+a²의 기울기와 헤시안을 만듭니다. 기존 임의의 Python 함수를 자동으로 모두 미분하는 방식이 아니므로 지원되는 연산으로 모델을 작성해야 합니다. 사용한 API는 [CasADi 공식 문서](https://web.casadi.org/docs/)에서 확인할 수 있습니다.

In [ ]:
import casadi as ca

z = ca.SX.sym("z", 2)
expr = ca.sin(z[0] * z[1]) + z[0] ** 2
adfun = ca.Function("adfun", [z], [expr, ca.gradient(expr, z), ca.hessian(expr, z)[0]])
out = adfun(point)
record("CasADi 기울기 비교", np.max(abs(np.array(out[1]).ravel() - exact_grad)), 1e-12)
print("CasADi 버전:", ca.__version__)

### 8.4.8 CasADi for Optimization — CasADi를 이용한 최적화

**교재 인쇄 pp.533–536**

함수와 도함수를 구성해 최적화 솔버에 전달합니다. 변수·비용·제약을 명시적으로 정의합니다. 솔버 종료 후 원래 문제의 잔차를 확인합니다.

보충 문제는 점 (1,0.5)와의 제곱 거리를 단위 원판 안에서 최소화합니다. CasADi가 비용·제약 도함수를 계산하고 SLSQP가 수치 최적화를 수행합니다. 해석적 방사 투영 결과와 비교합니다. §8.5에서도 같은 역할 분담으로 직접 최적 제어 문제를 구성합니다.

In [ ]:
# CasADi는 함수와 정확한 도함수를 구성하고 SciPy SLSQP가 NLP를 푼다.
w = ca.SX.sym("w", 2)
obj = (w[0] - 1) ** 2 + (w[1] - 0.5) ** 2
con = 1 - ca.dot(w, w)
cf = ca.Function("small_nlp", [w], [obj, ca.gradient(obj, w), con, ca.jacobian(con, w)])
res = minimize(
    lambda x: float(cf(x)[0]),
    [0.1, 0.1],
    jac=lambda x: np.array(cf(x)[1]).ravel(),
    method="SLSQP",
    constraints=[
        {
            "type": "ineq",
            "fun": lambda x: float(cf(x)[2]),
            "jac": lambda x: np.array(cf(x)[3]).ravel(),
        }
    ],
    options={"ftol": 1e-12, "maxiter": 100},
)
assert res.success
known = np.array([1.0, 0.5]) / np.sqrt(1.25)
record("CasADi 도함수 NLP 해 차이", np.max(abs(res.x - known)), 1e-6)

## 8.5 Direct Optimal Control Parameterizations — 직접 최적 제어의 매개화

**교재 인쇄 pp.536–538**

연속 제어 입력을 유한한 변수로 표현합니다. 상태 제거·연결·배치의 차이를 비교합니다. 같은 문제 비교와 다른 이산화 비교를 구분합니다.

보충 연속 모델의 비용은 ∫(x²+0.2u²)/2 dt + x(T)²입니다. 입력은 구간마다 상수이며 기본 N=20입니다. 단일·다중 사격법은 같은 RK4 전이와 RK4 비용 구적을 씁니다. 배치법은 음함수 중점 전이와 중점 구적을 쓰므로 유한 격자에서는 서로 다른 문제입니다. 별도 종단 강제약은 없습니다.

$$
\dot x=-x+0.2x^3+u,\quad J=\int_0^2\frac{x^2+0.2u^2}{2}\,dt+x(2)^2
$$

### 8.5.1 Direct Single Shooting — 직접 단일 사격법

**교재 인쇄 pp.538–540**

입력 열만 최적화 변수로 둡니다. 상태를 순서대로 전진 모사합니다. 상태 제약은 입력 열의 비선형 제약이 됩니다.

현재 상태에서 전체 입력 열로 궤적을 계산하고 비용과 상태 제약을 입력의 함수로 표현합니다. CasADi로 축약 비용 기울기와 상태 야코비안을 생성합니다. 변수는 적지만 초기 상태의 영향이 긴 구간을 통해 전파되므로 불안정 계에서는 민감도가 커질 수 있습니다.

In [ ]:
# OCP 생성기: 같은 RK4 모델을 단일/다중 사격법으로 각각 구성.
# 배치법은 1점 Gauss(음함수 중점)와 중점 구적을 사용한다.
def make_ocp(n=20, kind="single"):
    n = positive_integer(n, "구간 수")
    if kind not in {"single", "multiple", "midpoint"}:
        raise ValueError("알 수 없는 이산화 방법")
    h = 2 / n
    sx = ca.SX.sym("state")
    su = ca.SX.sym("input")

    def dyn(s, u):
        return -s + 0.2 * s**3 + u

    def running(s, u):
        return 0.5 * (s * s + 0.2 * u * u)

    k1 = dyn(sx, su)
    k2 = dyn(sx + h * k1 / 2, su)
    k3 = dyn(sx + h * k2 / 2, su)
    k4 = dyn(sx + h * k3, su)
    ph = sx + h * (k1 + 2 * k2 + 2 * k3 + k4) / 6
    lc = (
        h
        * (
            running(sx, su)
            + 2 * running(sx + h * k1 / 2, su)
            + 2 * running(sx + h * k2 / 2, su)
            + running(sx + h * k3, su)
        )
        / 6
    )
    step = ca.Function("step", [sx, su], [ph, lc])
    U = ca.SX.sym("U", n)
    param = ca.SX.sym("initial")
    X = [param]
    cost = 0
    defects = []
    if kind == "single":
        for k in range(n):
            nxt, cc = step(X[-1], U[k])
            X.append(nxt)
            cost += cc
        w = U
        lb = -np.ones(n)
        ub = np.ones(n)
    else:
        XX = ca.SX.sym("X", n)
        X = [param] + [XX[k] for k in range(n)]
        w = ca.vertcat(XX, U)
        lb = np.r_[np.full(n, -1.2), -np.ones(n)]
        ub = -lb
        for k in range(n):
            if kind == "multiple":
                nxt, cc = step(X[k], U[k])
                defects.append(nxt - X[k + 1])
                cost += cc
            else:
                mid = (X[k] + X[k + 1]) / 2
                defects.append(X[k] + h * dyn(mid, U[k]) - X[k + 1])
                cost += h * running(mid, U[k])
    cost += X[-1] ** 2
    xx = ca.vertcat(*X)
    dd = ca.vertcat(*defects) if defects else ca.SX.zeros(0, 1)
    cf = ca.Function(
        "ocp",
        [w, param],
        [cost, ca.gradient(cost, w), dd, ca.jacobian(dd, w), xx, ca.jacobian(xx, w)],
    )

    def solve(initial=0.8, guess=None):
        initial = finite_scalar(initial, "초기 상태")
        if abs(initial) > 1.2:
            raise ValueError("초기 상태가 상태 한계를 벗어났습니다.")
        if guess is None:
            guess = (
                np.zeros(n)
                if kind == "single"
                else np.r_[initial * np.exp(-np.arange(1, n + 1) * h), np.zeros(n)]
            )
        guess = finite_vector(guess, n if kind == "single" else 2 * n, "초기 추정")
        constraints = []
        if kind == "single":
            constraints = [
                {
                    "type": "ineq",
                    "fun": lambda v: np.r_[
                        1.2 - np.array(cf(v, initial)[4]).ravel()[1:],
                        1.2 + np.array(cf(v, initial)[4]).ravel()[1:],
                    ],
                    "jac": lambda v: np.vstack(
                        [
                            -np.array(cf(v, initial)[5])[1:],
                            np.array(cf(v, initial)[5])[1:],
                        ]
                    ),
                }
            ]
        else:
            constraints = [
                {
                    "type": "eq",
                    "fun": lambda v: np.array(cf(v, initial)[2]).ravel(),
                    "jac": lambda v: np.array(cf(v, initial)[3]),
                }
            ]
        result = minimize(
            lambda v: float(cf(v, initial)[0]),
            guess,
            jac=lambda v: np.array(cf(v, initial)[1]).ravel(),
            method="SLSQP",
            bounds=list(zip(lb, ub)),
            constraints=constraints,
            options={"ftol": 1e-11, "maxiter": 250},
        )
        if not result.success:
            raise RuntimeError((kind, n, result.message))
        if not np.all(np.isfinite(result.x)) or not np.isfinite(result.fun):
            raise RuntimeError("비유한 최적화 결과")
        vals = cf(result.x, initial)
        states = np.array(vals[4]).ravel()
        inputs = result.x[-n:]
        defect = float(np.max(abs(np.array(vals[2])))) if kind != "single" else 0.0
        assert (
            defect < 1e-7
            and max(abs(inputs)) <= 1 + 1e-8
            and max(abs(states)) <= 1.2 + 1e-8
        )
        return dict(
            result=result,
            x=states,
            u=inputs,
            cost=result.fun,
            defect=defect,
            n=n,
            kind=kind,
            cf=cf,
            step=step,
        )

    return solve


single = make_ocp(20, "single")()
print("단일 사격 비용:", single["cost"], "변수 수:", len(single["result"].x))

### 8.5.2 Direct Multiple Shooting — 직접 다중 사격법

**교재 인쇄 pp.540–544**

각 구간의 시작 상태도 변수로 둡니다. 짧은 적분 결과와 다음 상태를 등식으로 연결합니다. 동일한 RK4 문제의 단일 사격 해와 비교합니다.

초기 상태를 제외한 N개 상태와 N개 입력을 최적화합니다. 주 예제는 스칼라 상태여서 40변수입니다. 초기 추정 궤적은 동역학과 완전히 일치하지 않아도 되지만 마지막에는 연결 잔차를 만족해야 합니다. 같은 RK4 모델과 비용을 사용한 단일 사격법과 입력·비용을 비교합니다.

$$
c_k(X,U)=\Phi_h(x_k,u_k)-x_{k+1}=0
$$

In [ ]:
multiple = make_ocp(20, "multiple")()
record("단일·다중 사격 입력 차이", np.max(abs(single["u"] - multiple["u"])), 2e-5)
record("단일·다중 사격 비용 차이", abs(single["cost"] - multiple["cost"]), 1e-8)
record("다중 사격 동역학 잔차", multiple["defect"], 1e-8)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
tt = np.linspace(0, 2, 21)
axs[0].plot(tt, single["x"], label="단일 사격")
axs[0].plot(tt[::2], multiple["x"][::2], "o", ms=4, label="다중 사격")
axs[1].step(tt[:-1], single["u"], where="post", label="단일 사격")
axs[1].plot(tt[:-1:2], multiple["u"][::2], "o", ms=4, label="다중 사격")
for ax in axs:
    ax.set_xlabel("예측 시간")
    ax.legend()
axs[0].set(ylabel="상태", title="같은 이산 문제의 상태")
axs[1].set(ylabel="입력", title="같은 이산 문제의 입력")
savefig(fig, "shooting")

**결과 해석:** 같은 RK4 이산 문제의 두 표현이 수치 오차 안에서 일치합니다. 중간 최적화 반복이나 모든 비선형 문제의 결과가 같다는 뜻은 아닙니다.

### 8.5.3 Direct Transcription and Collocation Methods — 직접 전사와 배치법

**교재 인쇄 pp.544–548**

적분 방정식을 유한한 대수 제약으로 바꿉니다. 배치점과 구적법을 일관되게 선택합니다. 격자를 세분화하고 구간 내부도 확인합니다.

1점 Gauss 배치를 사용합니다. N=10,20,40에서 사격법과 배치법의 최적 비용 차이가 줄어드는지 확인합니다. 이는 선택한 이산화 사이의 일관성 검사이며 연속 문제의 전역 최적해 증명은 아닙니다. RK4 입력을 독립 고정밀 적분기로 다시 모사하여 구간 내부 표본도 상태 한계 안에 있는지 검사합니다.

$$
x_m=\frac{x_k+x_{k+1}}2,\quad x_{k+1}-x_k=hf_c(x_m,u_k),\quad L_k=h\ell_c(x_m,u_k)
$$

In [ ]:
collocation = make_ocp(20, "midpoint")()
record("배치법 동역학 잔차", collocation["defect"], 1e-8)
comparison = []
for n in [10, 20, 40]:
    ss = single if n == 20 else make_ocp(n, "single")()
    cc = collocation if n == 20 else make_ocp(n, "midpoint")()
    comparison.append((n, abs(ss["cost"] - cc["cost"]), np.max(abs(ss["u"] - cc["u"]))))
assert comparison[-1][1] < comparison[0][1]
record("세밀한 격자 사격·배치 비용 차이", comparison[-1][1], 2e-4)
# RK4 해의 구간 내부를 독립 고정밀 ODE로 다시 적분하여 표본 상태 제약 확인.
y = 0.8
dense = [y]
for u in single["u"]:
    integ = solve_ivp(
        lambda t, x: -x + 0.2 * x**3 + u,
        [0, 0.1],
        [y],
        rtol=1e-11,
        atol=1e-13,
        t_eval=np.linspace(0, 0.1, 11),
    )
    assert integ.success
    y = integ.y[0, -1]
    dense.extend(integ.y[0, 1:])
record("독립 연속 적분 종단 차이", abs(y - single["x"][-1]), 1e-6)
record("구간 내부 표본 상태 제한 위반", max(0.0, max(abs(np.array(dense))) - 1.2), 1e-8)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].loglog([r[0] for r in comparison], [r[1] for r in comparison], "o-")
axs[0].set(
    xlabel="구간 수 N", ylabel="최적 비용 차이", title="서로 다른 이산화의 격자 세분화"
)
axs[1].plot(np.linspace(0, 2, len(dense)), dense, label="독립 연속 적분")
axs[1].plot(np.linspace(0, 2, 21), collocation["x"], "o", ms=3, label="중점 배치 상태")
axs[1].set(xlabel="시간", ylabel="상태", title="구간 내부도 별도로 확인")
axs[1].legend()
savefig(fig, "collocation")

**결과 해석:** 두 이산화의 비용 차이가 격자 세분화로 줄어듭니다. 구간 내부 상태 검사도 유한 표본 검사임을 구분합니다.

## 8.6 Nonlinear Optimization — 비선형 최적화

**교재 인쇄 pp.548–549**

목적함수와 제약을 함께 선형화·이차화합니다. 원시 변수와 제약 승수를 함께 고려합니다. 라그랑지안의 곡률이 핵심 역할을 합니다.

동역학 제약을 포함한 최적 제어는 일반 NLP 형태로 볼 수 있습니다. 비용 헤시안과 라그랑지안 헤시안은 다를 수 있습니다. 정칙성과 이차 조건을 확인해야 KKT 해를 국소 최소와 연결할 수 있습니다.

$$
\mathcal L(w,\lambda,\mu)=F(w)+\lambda^TG(w)+\mu^TH(w)
$$

### 8.6.1 Optimality Conditions and Perturbation Analysis — 최적성 조건과 섭동 분석

**교재 인쇄 pp.549–552**

KKT는 비용과 제약의 최적성 조건을 모읍니다. 정칙한 해에서 파라미터 감도를 계산합니다. 활성 집합 변화나 퇴화에서는 주의가 필요합니다.

보충 등식 문제 min ||w||²/2, w₁+w₂=p에서 w=(p/2,p/2), λ=−p/2입니다. KKT 행렬로 해와 감도를 구해 해석식과 비교합니다. 부등식 활성 집합이 변할 때 해가 매끄럽게 미분 가능한지는 별도의 조건이 필요합니다.

$$
\nabla_w\mathcal L=0,\quad G=0,\quad H\le0,\quad\mu\ge0,\quad\mu_iH_i=0
$$

In [ ]:
# 등식 제약 보충 문제: min ||w||²/2, w1+w2=p. KKT 해와 파라미터 감도.
KKT = np.block([[np.eye(2), np.ones((2, 1))], [np.ones((1, 2)), np.zeros((1, 1))]])
param = 1.3
sol = np.linalg.solve(KKT, [0, 0, param])
ds = np.linalg.solve(KKT, [0, 0, 1])
record("KKT 파라미터 감도 차이", np.max(abs(ds - np.array([0.5, 0.5, -0.5]))), 1e-12)
record("KKT 원시·쌍대 잔차", np.max(abs(KKT @ sol - [0, 0, param])), 1e-12)

### 8.6.2 Nonlinear Optimization with Equalities — 등식 제약 비선형 최적화

**교재 인쇄 pp.552–553**

KKT 잔차에 뉴턴 방법을 적용합니다. 제약 야코비안과 라그랑지안 헤시안을 사용합니다. 최종 제약 잔차와 정지 잔차를 함께 봅니다.

보충 문제는 ((w₁−1)²+w₂²)/2를 w₂=w₁² 아래 최소화합니다. 라그랑지안은 비용+λ(w₂−w₁²)입니다. 정확한 KKT 야코비안으로 국소 뉴턴 반복을 수행하고 잔차를 검사합니다. 이 작은 예제는 제거한 비용의 이차 미분이 양수여서 해석적으로 최소를 확인할 수 있습니다.

$$
\begin{bmatrix}\nabla^2_{ww}\mathcal L&G_w^T\\G_w&0\end{bmatrix}\begin{bmatrix}\Delta w\\\Delta\lambda\end{bmatrix}=-\begin{bmatrix}\nabla_w\mathcal L\\G\end{bmatrix}
$$

In [ ]:
# 비선형 등식 w2=w1²: 정확한 라그랑지안 헤시안으로 KKT 뉴턴.
def kres(z):
    a, b, l = z
    return np.array([a - 1 - 2 * a * l, b + l, b - a * a])


def kjac(z):
    a, b, l = z
    return np.array([[1 - 2 * l, 0, -2 * a], [0, 1, 1], [-2 * a, 1, 0]])


z = np.array([0.6, 0.36, -0.36])
kh = []
for i in range(12):
    kh.append(np.linalg.norm(kres(z)))
    if kh[-1] < 1e-12:
        break
    z += np.linalg.solve(kjac(z), -kres(z))
record("비선형 등식 KKT 잔차", np.linalg.norm(kres(z)), 1e-10)
assert 1 + 6 * z[0] ** 2 > 0  # 제거한 목적함수의 이차 미분: 이 문제에서는 엄격 볼록.

### 8.6.3 Hessian Approximations — 헤시안 근사

**교재 인쇄 pp.553–556**

정확한 헤시안에는 잔차의 이차 미분이 들어갑니다. Gauss–Newton은 잔차 야코비안 곱을 사용합니다. 양의 준정부호성과 정확한 곡률은 다릅니다.

r=(w₁²−1,w₂−0.3)에서 잔차가 큰 점의 정확한 헤시안은 음의 고유값을 가질 수 있지만 JᵀJ는 양의 준정부호입니다. 잔차가 0인 점에서는 둘이 일치합니다. 일반 NLP에서는 비용의 최소제곱 구조 외에 제약 곡률과 BFGS 같은 근사도 고려합니다.

$$
\nabla^2F=J_r^TJ_r+\sum_i r_i\nabla^2r_i,\qquad B_{\mathrm{GN}}=J_r^TJ_r
$$

In [ ]:
# 비선형 최소제곱: 정확한 Hessian = JᵀJ + 잔차 가중 2차 미분.
z = ca.SX.sym("ls", 2)
residual = ca.vertcat(z[0] ** 2 - 1, z[1] - 0.3)
J = ca.jacobian(residual, z)
cost = 0.5 * ca.dot(residual, residual)
hfun = ca.Function("hessians", [z], [ca.hessian(cost, z)[0], J.T @ J])
he, hg = [np.array(v) for v in hfun([0.1, 0.0])]
assert np.linalg.eigvalsh(he).min() < 0 and np.linalg.eigvalsh(hg).min() >= 0
record(
    "잔차 영점 GN 헤시안 차이",
    np.max(abs(np.array(hfun([1, 0.3])[0]) - np.array(hfun([1, 0.3])[1]))),
    1e-12,
)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(np.arange(2) - 0.17, np.linalg.eigvalsh(he), 0.34, label="정확한 헤시안")
ax.bar(np.arange(2) + 0.17, np.linalg.eigvalsh(hg), 0.34, label="Gauss–Newton")
ax.axhline(0, color="gray")
ax.set(
    xticks=[0, 1],
    xticklabels=["작은 고유값", "큰 고유값"],
    ylabel="고유값",
    title="잔차가 클 때 두 곡률은 다름",
)
ax.legend()
savefig(fig, "hessians")

**결과 해석:** 정확한 곡률과 Gauss–Newton 근사는 큰 잔차에서 다릅니다. 양의 준정부호 근사가 정확한 뉴턴 곡률을 보존하는 것은 아닙니다.

## 8.7 Newton-Type Optimization with Inequalities — 부등식이 있는 뉴턴형 최적화

**교재 인쇄 pp.556–557**

활성 제약이 바뀌는 최적화를 다룹니다. SQP와 내부점법은 서로 다른 하위 문제를 풉니다. 원래 비선형 제약의 잔차를 확인합니다.

같은 단위 원판 위 거리 최소화 문제를 두 방법으로 풉니다. 목표는 (1.4,0.7)이며 해는 목표를 단위 원으로 방사 투영한 점입니다. 구현은 구조를 보여 주는 작은 교육용 반복이고 산업용 솔버의 속도 비교가 아닙니다.

### 8.7.1 Sequential Quadratic Programming — 순차 이차계획법

**교재 인쇄 pp.557–558**

현재 점에서 이차 하위 문제를 만듭니다. 비선형 제약을 일차 근사합니다. 하위 문제의 실행 가능성이 원래 제약을 보장하지 않습니다.

단위 원판 보충 문제에서 B=I인 양의 정부호 근사를 씁니다. CVXPY로 QP를 풀고 비용+제약 위반의 L1 메리트 함수로 단계 길이를 조절합니다. 중간 점이 비선형 제약을 위반할 수 있으므로 최종 원판 제약을 직접 검사합니다.

$$
\min_d\;\nabla F^Td+\frac12d^TBd\quad\text{s.t.}\quad H(w)+H_w(w)d\le0
$$

In [ ]:
# min ||z-target||²/2, ||z||²<=1. 양의 정부호 근사 Hessian을 쓰는 SQP.
target = np.array([1.4, 0.7])
exact = target / np.linalg.norm(target)


def obj(z):
    return 0.5 * np.sum((z - target) ** 2)


def ineq(z):
    return z @ z - 1


z = np.zeros(2)
sqp_path = [z.copy()]
sqp_lams = []
for it in range(60):
    delta = cp.Variable(2)
    constraint = ineq(z) + 2 * z @ delta <= 0
    sub = cp.Problem(
        cp.Minimize(0.5 * cp.sum_squares(delta) + (z - target) @ delta), [constraint]
    )
    sub.solve(solver=cp.CLARABEL, tol_gap_abs=1e-11, tol_feas=1e-11, tol_gap_rel=1e-11)
    assert sub.status == cp.OPTIMAL
    d = delta.value
    lam = float(constraint.dual_value)
    if np.linalg.norm(d) < 1e-9:
        break
    # L1 merit test; linearized constraint satisfaction alone is insufficient.
    rho = max(10.0, 2 * lam)
    merit = lambda y: obj(y) + rho * max(0.0, ineq(y))
    pred = (z - target) @ d - rho * max(0.0, ineq(z))
    alpha = 1.0
    while merit(z + alpha * d) > merit(z) + 1e-4 * alpha * pred:
        alpha *= 0.5
        if alpha < 1e-12:
            raise RuntimeError("SQP merit 탐색 실패")
    z += alpha * d
    sqp_path.append(z.copy())
    sqp_lams.append(lam)
sqp_sol = z.copy()
record("SQP 해석적 해 차이", np.max(abs(z - exact)), 2e-6)
record("SQP 최종 비선형 제약 위반", max(0.0, ineq(z)), 1e-7)

### 8.7.2 Nonlinear Interior Point Methods — 비선형 내부점법

**교재 인쇄 pp.558–560**

로그 장벽으로 경계 안쪽에서 문제를 풉니다. 장벽 계수를 줄이며 원래 해에 접근합니다. 유한 장벽 계수에서는 편향이 남습니다.

같은 원판 문제의 장벽 함수 F(z)−μ log(1−zᵀz)를 감쇠 뉴턴으로 풉니다. 초기점은 원점이고 모든 반복에서 양의 슬랙을 유지합니다. μ를 줄이면서 해석적 제약 최적해에 접근하는지 검사합니다. 이 코드는 로그 장벽의 기본 구조를 보이는 예이며 완전한 원시·쌍대 내부점 솔버는 아닙니다.

$$
\phi_\mu(w)=F(w)-\mu\sum_i\log(-H_i(w)),\qquad\lambda_i(-H_i)=\mu
$$

In [ ]:
# 같은 원판 문제의 로그 장벽: strictly interior start, damped Newton.
z = np.zeros(2)
ip_path = [z.copy()]
central = []
for mu in [1.0, 0.2, 0.04, 0.008, 0.0016, 0.00032, 1e-5, 1e-7]:
    for it in range(40):
        slack = 1 - z @ z
        grad = z - target + 2 * mu * z / slack
        hh = np.eye(2) + 2 * mu * np.eye(2) / slack + 4 * mu * np.outer(z, z) / slack**2
        step = np.linalg.solve(hh, -grad)
        if np.linalg.norm(grad) < 1e-9:
            break
        alpha = 1.0
        phi = lambda y: obj(y) - mu * np.log(1 - y @ y)
        while 1 - (z + alpha * step) @ (z + alpha * step) <= 0:
            alpha *= 0.5
        while phi(z + alpha * step) > phi(z) + 1e-4 * alpha * grad @ step:
            alpha *= 0.5
            if alpha < 1e-14:
                break
        z += alpha * step
        ip_path.append(z.copy())
    lam = mu / (1 - z @ z)
    central.append((mu, np.linalg.norm(z - exact), lam * (1 - z @ z)))
ip_sol = z.copy()
record("로그 장벽 최종 해 차이", np.max(abs(z - exact)), 2e-6)
record("장벽 상보성 등식 오차", max(abs(a - c) for a, b, c in central), 1e-12)
assert min(1 - v @ v for v in ip_path) > 0

### 8.7.3 Comparison of SQP and Nonlinear IP Methods — SQP와 비선형 내부점법 비교

**교재 인쇄 pp.560–561**

같은 목적함수와 제약으로 결과를 비교합니다. SQP의 선형 제약과 장벽의 내부 경로를 구분합니다. 작은 예제의 반복 수로 보편적 속도 순위를 매기지 않습니다.

두 반복의 경로와 로그 장벽 계수에 따른 해 편향을 표시합니다. 목표점의 정규화로 얻은 해석적 해를 공통 기준으로 사용합니다. 두 방법의 실제 성능은 희소성, 선형 대수, 초기화, 정확도, 문제 구조에 따라 달라집니다.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
theta = np.linspace(0, 2 * np.pi, 300)
axs[0].plot(np.cos(theta), np.sin(theta), color="gray")
for path, label in [(sqp_path, "SQP"), (ip_path, "로그 장벽")]:
    arr = np.array(path)
    axs[0].plot(arr[:, 0], arr[:, 1], ".-", label=label)
axs[0].plot(*exact, "x", ms=10, label="해석적 해")
axs[0].set(xlabel="변수 z₁", ylabel="변수 z₂", title="같은 제약 문제의 반복 경로")
axs[0].legend()
axs[1].loglog([r[0] for r in central], [r[1] for r in central], "o-")
axs[1].set(
    xlabel="장벽 계수 μ",
    ylabel="해석적 해와의 거리",
    title="유한 μ에서는 장벽 편향이 남음",
)
savefig(fig, "sqp_ip")

**결과 해석:** 같은 최적해를 향해도 반복 경로가 다릅니다. 장벽법은 계수가 유한할 때 원래 경계 해와 차이가 남습니다.

## 8.8 Structure in Discrete Time Optimal Control — 이산 최적 제어의 구조

**교재 인쇄 pp.561–562**

시간별 비용과 동역학은 국소적으로 연결됩니다. 이 구조를 사용하면 큰 행렬 계산을 줄일 수 있습니다. 상태 제거는 변수 수와 희소성을 함께 바꿉니다.

일반 NLP 형태를 그대로 조밀한 문제로 풀 수 있지만 시간 구조를 이용하면 계산을 크게 바꿀 수 있습니다. 이 절에서는 독립 선형 이차 모델을 사용해 같은 해가 서로 다른 선형 대수 표현으로 계산되는지 확인합니다.

### 8.8.1 Simultaneous Approach — 동시 접근의 구조

**교재 인쇄 pp.562–564**

동역학 등식은 인접 상태와 입력에만 의존합니다. 시간별 비용은 블록 구조를 만듭니다. 희소 야코비안 패턴을 직접 확인합니다.

§8.5 다중 사격법 최적점에서 연결 제약 야코비안을 표시합니다. 그림은 모든 상태 변수 뒤에 모든 입력 변수를 둔 순서입니다. 상태·입력을 시간 순서로 교차 배치하면 띠 모양을 더 뚜렷하게 볼 수 있지만 구조 자체는 같습니다.

In [ ]:
# 최적점에서 다중 사격 동역학 야코비안의 희소 패턴.
cf = multiple["cf"]
v = multiple["result"].x
JJ = np.array(cf(v, 0.8)[3])
assert JJ.shape == (20, 40)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.spy(JJ, markersize=4)
ax.set(
    xlabel="상태 변수 다음 입력 변수",
    ylabel="동역학 등식",
    title="각 등식은 인접 상태와 해당 입력에만 의존",
)
savefig(fig, "sparsity")

**결과 해석:** 각 등식의 비영 원소는 인접한 상태와 해당 시점 입력에만 나타납니다. 이 코드는 희소성 구조를 표시하며 전용 희소 NLP 솔버의 성능을 측정하지 않습니다.

### 8.8.2 Linear Quadratic Problems (LQP) — 선형 이차 문제

**교재 인쇄 pp.564**

선형화된 동역학과 이차 비용을 묶습니다. 양의 입력 곡률 아래 선형 시스템으로 풉니다. 독립 보충 모델로 세 해법을 비교합니다.

보충 모델은 A=[[1,0.1],[0,0.9]], B=[0.005,0.1]ᵀ, 초기 상태 (1,−0.2)입니다. 종단 비용은 이산 리카티 방정식의 해입니다. 이 비교에서는 입력·상태 강제약을 제거합니다. 추가 입력 제한은 §8.8.7에서 따로 넣습니다.

In [ ]:
# 독립 LQ 보충 모델. 제약을 제거하고 3개의 선형 대수 해법을 비교.
AA = np.array([[1.0, 0.1], [0.0, 0.9]])
BB = np.array([[0.005], [0.1]])
QQ = np.eye(2)
RR = np.array([[0.2]])
PP = solve_discrete_are(AA, BB, QQ, RR)
NN = 18
x0 = np.array([1.0, -0.2])
SX = np.vstack([np.linalg.matrix_power(AA, k + 1) for k in range(NN)])
SU = np.zeros((2 * NN, NN))
for i in range(NN):
    for j in range(i + 1):
        SU[2 * i : 2 * i + 2, j : j + 1] = np.linalg.matrix_power(AA, i - j) @ BB
QB = block_diag(*([QQ] * (NN - 1) + [PP]))
HH = SU.T @ QB @ SU + 0.2 * np.eye(NN)
FF = SU.T @ QB @ SX

### 8.8.3 LQP Solution by Riccati Recursion — 리카티 재귀로 LQP 풀기

**교재 인쇄 pp.564–566**

종단 비용에서 역방향으로 가치 곡률을 전달합니다. 각 단계의 작은 입력 시스템을 풉니다. 계산한 피드백으로 상태를 전진 모사합니다.

K에 음의 부호를 포함하는 u=Kx 관례를 사용합니다. 각 단계에서 큰 전체 KKT 대신 입력 차원의 선형 방정식을 풉니다. 일반 affine LQP에는 일차 항과 상수 보정도 있지만 이 실습은 원점 조절의 동차 LQ 비교입니다.

$$
K_k=-(R+B^TP_{k+1}B)^{-1}B^TP_{k+1}A,\quad P_k=Q+A^TP_{k+1}A+A^TP_{k+1}BK_k
$$

In [ ]:
Ps = [None] * (NN + 1)
Ks = [None] * NN
Ps[NN] = PP
for i in range(NN - 1, -1, -1):
    Ks[i] = -np.linalg.solve(RR + BB.T @ Ps[i + 1] @ BB, BB.T @ Ps[i + 1] @ AA)
    Ps[i] = QQ + AA.T @ Ps[i + 1] @ AA + AA.T @ Ps[i + 1] @ BB @ Ks[i]
x = x0.copy()
ur = []
for K in Ks:
    u = float((K @ x)[0])
    ur.append(u)
    x = AA @ x + BB[:, 0] * u
ur = np.array(ur)

### 8.8.4 LQP Solution by Condensing — 축약으로 LQP 풀기

**교재 인쇄 pp.566–568**

예측 상태를 입력 열의 함수로 제거합니다. 변수 수를 줄이는 대신 헤시안이 조밀해질 수 있습니다. 전체 KKT·리카티 해와 수치적으로 대조합니다.

상태 제거 행렬을 구성해 축약 입력 해를 계산합니다. 독립적으로 상태·입력과 동역학 등식을 유지한 전체 KKT 시스템도 풉니다. 세 해법의 입력 차이와 전체 동역학 잔차를 확인합니다. 역행렬 대신 선형 방정식 풀이를 사용합니다.

$$
X=S_xx_0+S_uU,\quad H=S_u^TQ_bS_u+R_b,\quad F=S_u^TQ_bS_x
$$

In [ ]:
uc = np.linalg.solve(HH, -FF @ x0)
# 전체 상태·입력 KKT 시스템은 동역학을 별도로 유지한다.
D = block_diag(QB, 0.2 * np.eye(NN))
EE = np.zeros((2 * NN, 3 * NN))
rhs = np.zeros(2 * NN)
for i in range(NN):
    EE[2 * i : 2 * i + 2, 2 * i : 2 * i + 2] = np.eye(2)
    EE[2 * i : 2 * i + 2, 2 * NN + i : 2 * NN + i + 1] = -BB
    if i:
        EE[2 * i : 2 * i + 2, 2 * i - 2 : 2 * i] = -AA
    else:
        rhs[:2] = AA @ x0
KK = np.block([[D, EE.T], [EE, np.zeros((2 * NN, 2 * NN))]])
sol = np.linalg.solve(KK, np.r_[np.zeros(3 * NN), rhs])
uk = sol[2 * NN : 3 * NN]
record("리카티·축약 입력 차이", np.max(abs(ur - uc)), 1e-10)
record("KKT·축약 입력 차이", np.max(abs(uk - uc)), 1e-10)
record("전체 KKT 동역학 잔차", np.max(abs(EE @ sol[: 3 * NN] - rhs)), 1e-10)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(uc, label="축약")
ax.plot(ur, "o", ms=4, label="리카티")
ax.plot(uk, "x", label="전체 KKT")
ax.set(xlabel="예측 단계", ylabel="입력", title="같은 LQ 문제의 세 해법")
ax.legend()
savefig(fig, "lq_solutions")

**결과 해석:** 리카티·축약·전체 KKT는 같은 무제약 LQ 문제를 서로 다른 계산 구조로 풀며 입력이 수치 오차 안에서 일치합니다.

### 8.8.5 Sequential Approaches and Sparsity Exploitation — 순차 접근과 희소성 활용

**교재 인쇄 pp.568–570**

상태를 제거해도 전파 구조를 이용할 수 있습니다. 수반을 역방향으로 전달해 기울기를 구합니다. 전체 민감도 행렬을 만들 필요가 없는 경우가 있습니다.

같은 선형 문제에서 종단 수반을 P x_N으로 놓고 역방향으로 기울기를 계산합니다. 축약 행렬의 HU+Fx₀와 비교합니다. 실습에서는 검증을 위해 두 표현을 모두 만들지만 실제 구현에서 항상 둘 다 구성할 필요는 없습니다.

$$
\lambda_N=Px_N,\quad g_k=Ru_k+B^T\lambda_{k+1},\quad\lambda_k=Qx_k+A^T\lambda_{k+1}
$$

In [ ]:
# 순방향 민감도 행렬을 만들지 않고 비용 수반을 뒤로 전달한다.
xs = (SX @ x0 + SU @ uc).reshape(NN, 2)
adj = PP @ xs[-1]
grad = np.zeros(NN)
for i in range(NN - 1, -1, -1):
    grad[i] = 0.2 * uc[i] + (BB.T @ adj)[0]
    if i:
        adj = QQ @ xs[i - 1] + AA.T @ adj
record("수반·축약 기울기 차이", np.max(abs(grad - (HH @ uc + FF @ x0))), 1e-10)
print(
    "전체 KKT 크기:",
    KK.shape,
    "축약 Hessian 크기:",
    HH.shape,
    "축약 Hessian 비영 원소:",
    np.count_nonzero(abs(HH) > 1e-12),
)

### 8.8.6 Differential Dynamic Programming — 미분 동적 계획법

**교재 인쇄 pp.570–572**

역방향 이차 근사에서 입력 보정과 피드백을 얻습니다. 비선형 전진 모사에 상태 편차 피드백을 사용합니다. Gauss–Newton 변형과 정확한 DDP를 구분합니다.

별도 이산 모델 x⁺=0.85x+0.08x³+0.3u, N=12, 초기 상태 1을 사용합니다. 비용은 Σ(x²+0.2u²)/2+x_N²이고 이 실습에는 강제약이 없습니다. 동역학 이차 미분을 생략한 Gauss–Newton DDP인 iLQR를 구현합니다. 비선형 전진 모사의 실제 비용 감소를 확인하고 독립 BFGS 해와 비용·기울기를 비교합니다. 정확한 헤시안을 쓰는 DDP의 수렴률을 이 구현에 그대로 주장하지 않습니다.

$$
u_k^{\mathrm{new}}=\bar u_k+\alpha k_k+K_k(x_k^{\mathrm{new}}-\bar x_k),\quad x_{k+1}^{\mathrm{new}}=f(x_k^{\mathrm{new}},u_k^{\mathrm{new}})
$$

In [ ]:
# DDP의 Gauss–Newton 변형(iLQR): 비선형 전진 모사에 상태 피드백을 사용.
# 별도 이산 모델이며 이 실습에는 입력·상태 강제약이 없다.
NIL = 12
rho = 0.2
terminal_weight = 2.0


def fd(x, u):
    return 0.85 * x + 0.08 * x**3 + 0.3 * u


def rollout(initial, U):
    X = [float(initial)]
    for u in U:
        X.append(fd(X[-1], u))
    return np.array(X)


def j_il(X, U):
    return 0.5 * (X[:-1] @ X[:-1] + rho * (U @ U) + terminal_weight * X[-1] ** 2)


def il_grad(initial, U):
    X = rollout(initial, U)
    adj = terminal_weight * X[-1]
    g = np.zeros(len(U))
    for k in range(len(U) - 1, -1, -1):
        g[k] = rho * U[k] + 0.3 * adj
        adj = X[k] + (0.85 + 0.24 * X[k] ** 2) * adj
    return g


U = np.zeros(NIL)
X = rollout(1.0, U)
hist = [j_il(X, U)]
for iteration in range(50):
    Vx = terminal_weight * X[-1]
    Vxx = terminal_weight
    ks = np.zeros(NIL)
    Ks = np.zeros(NIL)
    for k in range(NIL - 1, -1, -1):
        aa = 0.85 + 0.24 * X[k] ** 2
        bb = 0.3
        qx = X[k] + aa * Vx
        qu = rho * U[k] + bb * Vx
        qxx = 1 + aa * Vxx * aa
        quu = rho + bb * Vxx * bb
        qux = bb * Vxx * aa
        ks[k] = -qu / quu
        Ks[k] = -qux / quu
        Vx = qx + Ks[k] * quu * ks[k] + Ks[k] * qu + qux * ks[k]
        Vxx = qxx + Ks[k] ** 2 * quu + 2 * Ks[k] * qux
    if np.max(abs(il_grad(1.0, U))) < 1e-8:
        break
    accepted = False
    for alpha in 2.0 ** (-np.arange(16)):
        Xn = [1.0]
        Un = []
        for k in range(NIL):
            un = U[k] + alpha * ks[k] + Ks[k] * (Xn[-1] - X[k])
            Un.append(un)
            Xn.append(fd(Xn[-1], un))
        Xn = np.array(Xn)
        Un = np.array(Un)
        new = j_il(Xn, Un)
        if new < hist[-1]:
            X, U = Xn, Un
            hist.append(new)
            accepted = True
            break
    if not accepted:
        break
reference = minimize(
    lambda u: j_il(rollout(1.0, u), u),
    np.zeros(NIL),
    jac=lambda u: il_grad(1.0, u),
    method="BFGS",
    options={"gtol": 1e-9, "maxiter": 300},
)
record("iLQR 최종 기울기", np.max(abs(il_grad(1.0, U))), 1e-6)
record("iLQR·독립 최적화 비용 차이", abs(hist[-1] - reference.fun), 1e-8)
assert np.all(np.diff(hist) <= 0)
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(hist, "o-")
axs[0].set(
    xlabel="iLQR 반복", ylabel="실제 비선형 비용", title="피드백 전진 모사와 감소 검사"
)
axs[1].plot(X, label="iLQR 상태")
axs[1].plot(rollout(1.0, reference.x), "x", label="독립 최적화")
axs[1].set(xlabel="예측 단계", ylabel="상태", title="국소 해의 수치 일치")
axs[1].legend()
savefig(fig, "ilqr")

**결과 해석:** 비선형 전진 모사에서 실제 비용을 검사합니다. iLQR의 국소 해는 독립 최적화 결과와 비용 수준에서 일치합니다.

#### 보충 실습: 최적점 밖에서 수반 기울기를 검증하는 Taylor 검사

iLQR와 BFGS가 같은 `il_grad`를 사용하므로 두 최적 비용의 일치만으로 그 도함수 구현을 독립적으로 검증할 수는 없습니다. 특히 최적점에서는 실제 기울기 자체가 작습니다. 여기서는 초기 상태 0.8, 비최적 입력 열 $U_k=-0.1+0.15\sin(k)$에서 **비용만 재계산하는 검사**를 추가합니다.

단위 방향 $d$와 작은 양수 $h$에 대해
$$R_0(h)=|J(U+hd)-J(U)|=O(h),$$
$$R_1(h)=|J(U+hd)-J(U)-h\nabla J(U)^Td|=O(h^2).$$
$R_0$의 일차 항이 사라지지 않도록 $d=\nabla J/\|\nabla J\|$를 사용합니다. 수반 재귀에서 $f_x=0.85+0.24x^2$의 두 번째 항을 **의도적으로 누락한 오답**도 같은 검사에 넣습니다. 잘못된 기울기의 나머지는 일반적으로 일차로만 감소합니다. 모든 입력 좌표의 중앙차분도 별도로 대조하므로 한 방향에서 우연히 오차가 상쇄되는 경우를 보완합니다.

**읽는 법:** 로그–로그 기울기는 충분히 작은 $h$에서 올바른 보정은 약 2, 미보정·오답 보정은 약 1이어야 합니다. 너무 작은 간격에서는 부동소수점 상쇄가 나타날 수 있어 $10^{-2}$–$10^{-5}$ 구간에서 차수를 추정합니다. 이 검사는 구현한 이산 비용의 일차 미분을 검사합니다. 연속 시스템의 미분 정확도나 헤시안 정확도, 전역 최적성을 보장하지 않습니다.


In [ ]:
# 비정상점에서 비용 평가만으로 수반 미분을 독립 검증한다.
test_initial = 0.8
test_plan = -0.1 + 0.15 * np.sin(np.arange(NIL))
test_gradient = il_grad(test_initial, test_plan)
assert np.linalg.norm(test_gradient) > 0.1
test_direction = test_gradient / np.linalg.norm(test_gradient)


def test_cost(plan):
    return j_il(rollout(test_initial, plan), plan)


# 의도적인 오답: 비선형 동역학의 도함수 0.24*x²를 누락한다.
test_states = rollout(test_initial, test_plan)
bad_adjoint = terminal_weight * test_states[-1]
bad_gradient = np.zeros(NIL)
for k in range(NIL - 1, -1, -1):
    bad_gradient[k] = rho * test_plan[k] + 0.3 * bad_adjoint
    bad_adjoint = test_states[k] + 0.85 * bad_adjoint

steps = np.logspace(-1, -7, 13)
base_cost = test_cost(test_plan)
changes = np.array([test_cost(test_plan + h * test_direction) - base_cost for h in steps])
raw_remainder = abs(changes)
correct_remainder = abs(changes - steps * (test_gradient @ test_direction))
bad_remainder = abs(changes - steps * (bad_gradient @ test_direction))
fit_window = (steps <= 1.01e-2) & (steps >= 0.99e-5)
orders = [np.polyfit(np.log(steps[fit_window]), np.log(values[fit_window]), 1)[0]
          for values in (raw_remainder, correct_remainder, bad_remainder)]
assert 0.95 < orders[0] < 1.05
assert 1.95 < orders[1] < 2.05
assert 0.9 < orders[2] < 1.15
fd_step = 1e-6
basis = np.eye(NIL)
fd_gradient = np.array([(test_cost(test_plan + fd_step * e) -
                         test_cost(test_plan - fd_step * e)) / (2 * fd_step)
                        for e in basis])
record("비최적점 수반·중앙차분 기울기 오차", np.max(abs(test_gradient - fd_gradient)), 1e-7)
record("Taylor 미보정 관측 차수", orders[0])
record("Taylor 수반 보정 관측 차수", orders[1])
record("Taylor 오답 보정 관측 차수", orders[2])
record("동역학 도함수 누락의 기울기 오차", np.max(abs(bad_gradient - fd_gradient)))
assert np.max(abs(bad_gradient - fd_gradient)) > 1e-2
fig, ax = plt.subplots(figsize=(8, 4))
for values, label, order in zip(
    (raw_remainder, correct_remainder, bad_remainder),
    ("미보정 R₀", "수반 보정 R₁", "도함수 누락 오답"), orders
):
    ax.loglog(steps, values, "o-", label=f"{label}: 관측 차수 {order:.2f}")
ax.set(xlabel="섭동 크기 h", ylabel="비용 Taylor 나머지", title="최적점 밖의 도함수 검증")
ax.legend()
savefig(fig, "adjoint_taylor")


### 8.8.7 Additional Constraints in Optimal Control — 최적 제어의 추가 제약

**교재 인쇄 pp.572–573**

경로·종단 제약은 승수와 KKT 항을 추가합니다. 시간 국소 제약은 많은 희소 구조를 유지합니다. 무제약 리카티 입력을 단순 적용할 수 없습니다.

§8.8 LQ 문제에 입력 한계 0.4를 추가하여 QP로 풉니다. 무제약 최적 입력이 한계를 넘는지, 제약 최적해가 입력 한계를 만족하는지 확인합니다. 무제약 입력을 단순 포화시킨 값이 제약 MPC의 최적 입력 열과 같다고 보장할 수 없습니다.

In [ ]:
# 같은 LQ 모델에 입력 제약을 추가하면 무제약 리카티 식만으로는 해를 얻지 못한다.
u = cp.Variable(NN)
constrained = cp.Problem(
    cp.Minimize(0.5 * cp.quad_form(u, HH) + (FF @ x0) @ u), [cp.abs(u) <= 0.4]
)
constrained.solve(
    solver=cp.CLARABEL, tol_gap_abs=1e-10, tol_feas=1e-10, tol_gap_rel=1e-10
)
assert constrained.status == cp.OPTIMAL
record("추가 입력 제약 위반", max(0.0, np.max(abs(u.value)) - 0.4), 1e-8)
assert np.max(abs(uc)) > 0.4

#### 보충 실습 — 무제약 입력 열의 단순 포화와 제약 QP

§8.8.7의 같은 2상태 LQ 문제에서 입력 한계 0.4를 적용합니다. 무제약 최적 입력 열을 성분별로 자른 $U_{\rm clip}$와 입력 제한을 포함해 다시 최적화한 $U_{\rm QP}$를 비교합니다.
둘 다 입력 제약을 만족하지만, 미래 상태와 비용에 미치는 효과를 함께 조정한 QP 해가 더 낮은 비용을 가집니다. 축약 목적함수의 초기 상태 상수항은 양쪽에서 같으므로 비용 차이는 그대로 비교할 수 있습니다.

여기서 포화하는 대상은 **현재 상태에서 구한 전체 무제약 입력 열**입니다. 매 단계 리카티 피드백을 재평가하여 포화시키는 별도 폐루프 정책과 구분합니다. 원본의 특정 수치 예제를 재현하는 실습은 아닙니다.

In [ ]:
def compare_clipping():
    clipped = np.clip(uc, -0.4, 0.4)
    optimal = np.asarray(constrained.variables()[0].value).ravel()

    def reduced_cost(plan):
        return 0.5 * plan @ HH @ plan + (FF @ x0) @ plan

    gap = reduced_cost(clipped) - reduced_cost(optimal)
    record("입력 열 단순 포화의 추가 비용", gap)
    record("단순 포화와 제약 QP 입력 차이", np.max(abs(clipped - optimal)))
    record("단순 포화 입력 제한 위반", max(0.0, max(abs(clipped)) - 0.4), 1e-12)
    assert gap > 1e-5
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.7))
    for plan, label in [
        (uc, "무제약 최적"),
        (clipped, "단순 포화"),
        (optimal, "제약 QP"),
    ]:
        axs[0].plot(plan, "o-", ms=3, label=label)
        states = np.vstack([x0, (SX @ x0 + SU @ plan).reshape(NN, 2)])
        axs[1].plot(states[:, 0], label=label)
    axs[0].axhline(0.4, color="gray", ls="--")
    axs[0].axhline(-0.4, color="gray", ls="--")
    axs[0].set(xlabel="예측 단계", ylabel="입력", title="같은 한계, 서로 다른 입력 열")
    axs[0].legend()
    axs[1].set(
        xlabel="예측 단계",
        ylabel="첫 번째 상태",
        title=f"단순 포화의 추가 비용 {gap:.4f}",
    )
    axs[1].legend()
    savefig(fig, "clipping_cost")


compare_clipping()

## 8.9 Online Optimization Algorithms — 온라인 최적화 알고리즘

**교재 인쇄 pp.573–574**

매 측정마다 비슷한 최적화 문제가 반복됩니다. 이전 해와 분해·도함수 정보를 활용할 수 있습니다. 유한 반복 오차를 제어 결과와 함께 확인합니다.

온라인 MPC에서는 문제를 처음부터 끝까지 독립적으로 풀기보다 이전 계산을 다음 문제로 전달할 수 있습니다. 계산 지연, 측정 시점, 허용오차 및 실패 대응이 실제 제어와 연결됩니다. 이 노트북은 실제 장치의 마감시간이나 지연을 측정하지 않습니다.

### 8.9.1 General Algorithmic Considerations — 온라인 계산의 일반 고려사항

**교재 인쇄 pp.574–577**

이전 해를 다음 문제의 초기값으로 사용합니다. 시간이 진행하면 입력 열을 이동할 수 있습니다. 준비 계산과 측정 후 계산을 구분할 수 있습니다.

§8.8.6 이산 모델에 입력 한계 1.2를 두고 초기 상태를 1에서 0.1까지 바꿉니다. 영 초기화와 이전 해를 한 단계 이동한 초기화의 SLSQP 반복 수를 비교합니다. 같은 비용에 도달하는지 검사하며, 특정 예제의 반복 수를 모든 문제의 속도 보장으로 일반화하지 않습니다.

In [ ]:
# 고정 시드/모델의 이동 초기화. 반복 횟수는 솔버·문제에 의존한다.
def full_nmpc(initial, guess):
    r = minimize(
        lambda u: j_il(rollout(initial, u), u),
        guess,
        jac=lambda u: il_grad(initial, u),
        method="SLSQP",
        bounds=[(-1.2, 1.2)] * NIL,
        options={"ftol": 1e-10, "maxiter": 150},
    )
    if not r.success:
        raise RuntimeError(r.message)
    return r


initials = np.linspace(1.0, 0.1, 16)
warm = np.zeros(NIL)
warm_counts = []
cold_counts = []
diffs = []
for x in initials:
    wr = full_nmpc(x, warm)
    cr = full_nmpc(x, np.zeros(NIL))
    warm_counts.append(wr.nit)
    cold_counts.append(cr.nit)
    diffs.append(abs(wr.fun - cr.fun))
    warm = np.r_[wr.x[1:], 0.0]
record("이동·영 초기화 비용 차이", max(diffs), 1e-8)
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(cold_counts, "o-", label="매번 영 초기화")
ax.plot(warm_counts, "o-", label="이전 해 이동")
ax.set(
    xlabel="서로 다른 초기 상태의 문제 번호",
    ylabel="SLSQP 반복 횟수",
    title="이 보충 문제에서 관측한 반복 수",
)
ax.legend()
savefig(fig, "warm_start")

**결과 해석:** 같은 비용에 도달한 두 초기화의 반복 수를 비교합니다. 이동 초기화의 이점은 문제와 솔버에 따라 달라집니다.

### 8.9.2 Continuation Methods and Real-Time Iterations — 연속 추적과 실시간 반복

**교재 인쇄 pp.577–580**

파라미터가 바뀌는 동안 근이나 최적해를 추적합니다. 실시간 반복은 측정마다 소수의 갱신만 수행합니다. 작은 변화·정칙성·좋은 초기화 조건을 함께 봅니다.

보충 실습은 순차 Gauss–Newton 실시간 반복입니다. 입력 열을 중심으로 상태 민감도를 전파하고 측정마다 입력 제약을 포함한 QP 하나를 풉니다. 상태는 비선형 전진 모사로 재생성하며 상태 강제약은 없습니다. 동일한 상태에서 수렴시킨 최적 비용과 차이를 기록합니다. 이는 동시 다중 사격 RTI의 중간 결함을 재현한 구현이 아닙니다. 마지막에 z²=p 근 추적 예제로 파라미터 변화 크기와 뉴턴 1회의 오차 관계도 확인합니다. 관측한 수렴은 임의의 NMPC에 대한 안정성 보장이 아닙니다.

$$
R_z(p_{k+1},z_k)\Delta z_k=-R(p_{k+1},z_k),\quad z_{k+1}=z_k+\Delta z_k
$$

In [ ]:
# 순차 Gauss–Newton 실시간 반복: 측정당 QP 하나 + 비선형 재모사.
# 전체 입력 제약을 QP에 유지하며 상태 강제약은 이 모델에 두지 않는다.
def one_gn(initial, U):
    X = rollout(initial, U)
    S = np.zeros((NIL + 1, NIL))
    for k in range(NIL):
        S[k + 1] = (0.85 + 0.24 * X[k] ** 2) * S[k]
        S[k + 1, k] += 0.3
    weights = np.r_[np.ones(NIL), terminal_weight]
    Hgn = S.T @ (weights[:, None] * S) + rho * np.eye(NIL)
    grad = S.T @ (weights * X) + rho * U
    d = cp.Variable(NIL)
    qp = cp.Problem(
        cp.Minimize(0.5 * cp.quad_form(d, Hgn) + grad @ d), [cp.abs(U + d) <= 1.2]
    )
    qp.solve(solver=cp.CLARABEL, tol_gap_abs=1e-10, tol_feas=1e-10, tol_gap_rel=1e-10)
    if qp.status != cp.OPTIMAL:
        raise RuntimeError(qp.status)
    return U + d.value


xr = 1.0
xf = 1.0
warm = np.zeros(NIL)
fullwarm = np.zeros(NIL)
rstates = [xr]
fstates = [xf]
ru = []
fu = []
gaps = []
for k in range(30):
    plan = one_gn(xr, warm)
    best = full_nmpc(xr, plan)
    gaps.append(j_il(rollout(xr, plan), plan) - best.fun)
    full = full_nmpc(xf, fullwarm)
    ur = plan[0]
    uf = full.x[0]
    xr = fd(xr, ur)
    xf = fd(xf, uf)
    rstates.append(xr)
    fstates.append(xf)
    ru.append(ur)
    fu.append(uf)
    warm = np.r_[plan[1:], 0.0]
    fullwarm = np.r_[full.x[1:], 0.0]
record("1회 GN 폐루프 최종 상태", abs(xr), 1e-4)
record("1회 GN 입력 제한 위반", max(0.0, max(abs(np.array(ru))) - 1.2), 1e-8)
record("1회 GN의 최대 최적 비용 차이", max(gaps))
fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
axs[0].plot(rstates, label="측정당 QP 1회")
axs[0].plot(fstates, "--", label="수렴까지 최적화")
axs[0].set(xlabel="시간 단계", ylabel="상태", title="유한 반복 폐루프: 이 모델의 결과")
axs[0].legend()
axs[1].semilogy(np.maximum(gaps, 1e-15), "o-")
axs[1].set(
    xlabel="시간 단계",
    ylabel="같은 상태의 최적 비용과 차이",
    title="한 번의 반복은 정확한 최적해가 아님",
)
savefig(fig, "rti")
# 단순 파라미터 근 추적: 작은 변화가 뉴턴 1회의 추적 정확도를 높이는 예.
tracking = []
for count in [10, 20, 40, 80]:
    z = 1.0
    errors = []
    for p in np.linspace(1, 2, count + 1)[1:]:
        z -= (z * z - p) / (2 * z)
        errors.append(abs(z - np.sqrt(p)))
    tracking.append(max(errors))
assert tracking[-1] < tracking[0] / 30
record("작은 파라미터 변화의 뉴턴 추적 오차", tracking[-1], 3e-5)

**결과 해석:** 한 번의 QP로도 이 보충 모델의 폐루프는 수렴하지만 정확한 최적해와 차이가 남습니다. 같은 상태의 최적 비용을 별도로 계산해 그 차이를 표시합니다. 그래프의 로그 축에서는 1e−15 이하 차이를 1e−15로 표시합니다.

## 8.10 Notes — 문헌 해설

**교재 인쇄 pp.580–583**

적분·미분·최적화의 연구 흐름을 연결합니다. 전용 솔버의 성능은 구현과 문제 구조에 의존합니다. 교재의 역사적 서술과 현재 API를 구분합니다.

교재의 참고문헌은 직접 최적 제어, DAE, 자동 미분, SQP·내부점법, 구조적 선형 대수, 실시간 반복으로 이어집니다. 2017년 도구 설명을 현재의 성능 순위로 읽지 않습니다. CasADi 관련 코드는 공식 문서의 SX·Function·gradient·jacobian·hessian API를 사용합니다.

## 8.11 Exercises — 연습문제와 추가 탐구

**교재 인쇄 pp.583–595**

교재는 근 찾기부터 실시간 NMPC까지 연결합니다. 모델·격자·도함수·초기화의 영향을 분리합니다. 수치 성공과 수학적 보장의 차이를 설명합니다.

교재 전체 연습문제의 해답은 제공하지 않습니다.

**추가 과제**
1. RK4의 간격을 늘려 정확도와 안정성이 언제 다른 문제인지 설명하세요.
2. DAE 초기값을 대수 제약 밖에 놓았을 때 초기화 절차를 설계하세요.
3. 전진·역방향 AD의 계산량을 입력·출력 차원 관점에서 비교하세요.
4. 사격법과 배치법의 격자를 더 세분화하고 구간 내부 제약을 다시 검사하세요.
5. 무제약 LQ의 리카티 해를 포화시키는 방식과 제약 QP 해를 비교하세요.
6. 실시간 반복에서 초기 상태를 크게 바꾸고 비용 차이와 제약을 확인하세요.

**힌트:** 솔버 성공 플래그 외에 원래 잔차를 검사하세요. RTI에 상태 제약을 추가할 때는 선형화 제약만으로 실제 비선형 궤적의 실행 가능성을 보장할 수 없습니다.

In [ ]:
for name, value in CHECKS.items():
    print(f"{name}: {value:.9g}")
print("검증 지표 수:", len(CHECKS))
print(
    {
        "Python": sys.version.split()[0],
        "NumPy": np.__version__,
        "SciPy": scipy.__version__,
        "Matplotlib": matplotlib.__version__,
        "CVXPY": cp.__version__,
        "CasADi": ca.__version__,
    }
)

## 참고 자료

- Rawlings, Mayne, Diehl (2017), *Model Predictive Control: Theory, Computation, and Design*, 2판 1쇄, Chapter 8, pp.491–600.
- [CasADi 공식 문서](https://web.casadi.org/docs/): SX·Function과 도함수 구성 API를 확인했습니다.
- 모든 실행 예제는 직접 설계한 보충 문제입니다. 한글 그래프는 Google Fonts의 나눔고딕을 사용합니다.

원본 재현: 예제 8.1 (pp.495–496), 8.3 (pp.504–505), 8.4 (pp.511–512), 8.5 (pp.516–517), 8.9·8.11·8.12 (pp.522–531). 원본 모델과 알고리즘을 사용하되 그림의 전체 격자 범위를 동일하게 복제하지는 않습니다.